# Bloc 12 — Big Data & Scalable Pipelines
## Traitement distribué, Spark et montée en charge

## Étape 1 — Architecture Big Data

### Partie A — Mise en pratique

Le dataset énergétique actuel contient quelques centaines de milliers
d'observations.

À cette échelle, Pandas et PostgreSQL restent suffisamment efficaces
pour le pipeline principal.

Spark n'est donc pas introduit pour remplacer artificiellement les
outils existants.

Il est utilisé comme environnement expérimental permettant d'étudier :

- le traitement distribué ;
- le partitionnement ;
- les transformations paresseuses ;
- les agrégations parallèles ;
- les pipelines ML distribués ;
- la montée en charge vers plusieurs millions de lignes.

L'architecture devient :

$$
\text{PostgreSQL / Joblib}
\rightarrow
\text{Spark}
\rightarrow
\text{Transformations distribuées}
\rightarrow
\text{ML / Analytics}.
$$

Le pipeline principal PostgreSQL + FastAPI + Streamlit reste inchangé.

In [137]:
# ==========================================
# FORCE PYSPARK TO USE PYTHON 3.11
# ==========================================

import os
import sys


PYTHON_311 = (
    r"C:\Users\thiba\AppData\Local\Programs\Python\Python311\python.exe"
)


os.environ[
    "PYSPARK_PYTHON"
] = PYTHON_311

os.environ[
    "PYSPARK_DRIVER_PYTHON"
] = PYTHON_311


print(
    "Notebook Python :",
    sys.executable
)

print(
    "PySpark worker Python :",
    os.environ[
        "PYSPARK_PYTHON"
    ]
)

print(
    "PySpark driver Python :",
    os.environ[
        "PYSPARK_DRIVER_PYTHON"
    ]
)

Notebook Python : c:\Users\thiba\AppData\Local\Programs\Python\Python311\python.exe
PySpark worker Python : C:\Users\thiba\AppData\Local\Programs\Python\Python311\python.exe
PySpark driver Python : C:\Users\thiba\AppData\Local\Programs\Python\Python311\python.exe


In [138]:
# ==========================================
# BLOCK 12 — INITIALISATION
# ==========================================

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import time


PROJECT_DIR = Path(
    r"C:\Users\thiba\OneDrive\Bureau\Projet energie"
)

DATA_DIR = (
    PROJECT_DIR
    / "data_processed"
)

MODEL_DIR = (
    PROJECT_DIR
    / "models"
)


BLOCK12_DIR = (
    DATA_DIR
    / "block12"
)


BLOCK12_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


RANDOM_STATE = 42


print(
    "Projet :",
    PROJECT_DIR
)

print(
    "Block 12 :",
    BLOCK12_DIR
)

Projet : C:\Users\thiba\OneDrive\Bureau\Projet energie
Block 12 : C:\Users\thiba\OneDrive\Bureau\Projet energie\data_processed\block12


In [139]:
# ==========================================
# BIG DATA CONFIGURATION
# ==========================================

USE_SPARK_FOR_PRODUCTION = False

RUN_SPARK_LAB = True

KEEP_POSTGRESQL_PIPELINE = True


big_data_config = {

    "spark_production":
        USE_SPARK_FOR_PRODUCTION,

    "spark_lab":
        RUN_SPARK_LAB,

    "postgresql_pipeline":
        KEEP_POSTGRESQL_PIPELINE,

    "uses_2026":
        False,
}


display(
    pd.Series(
        big_data_config,
        name="Block 12 configuration",
    )
)

spark_production       False
spark_lab               True
postgresql_pipeline     True
uses_2026              False
Name: Block 12 configuration, dtype: bool

### Choix d'architecture

Le volume actuel ne justifie pas l'utilisation obligatoire d'un moteur
distribué.

Pour quelques centaines de milliers de lignes :

$$
N \approx 2.3\times10^5,
$$

Pandas et PostgreSQL sont généralement suffisants sur une machine
personnelle.

Spark devient surtout intéressant lorsque :

- les données ne tiennent plus confortablement en mémoire ;
- plusieurs millions ou milliards d'observations doivent être traitées ;
- le calcul doit être distribué sur plusieurs machines ;
- plusieurs traitements lourds doivent être parallélisés.

Le choix retenu est donc :

$$
Spark_{production}=OFF
$$

et :

$$
Spark_{laboratoire}=ON.
$$

In [140]:
# ==========================================
# PYSPARK CHECK
# ==========================================

try:

    import pyspark

    PYSPARK_AVAILABLE = True

    print(
        "PySpark disponible :",
        pyspark.__version__
    )

except ImportError:

    PYSPARK_AVAILABLE = False

    print(
        "PySpark non installé."
    )

PySpark disponible : 4.2.0


In [141]:
import sys


print(
    "Python notebook :",
    sys.executable
)

Python notebook : c:\Users\thiba\AppData\Local\Programs\Python\Python311\python.exe


In [142]:
# ==========================================
# SPARK SESSION
# ==========================================

if PYSPARK_AVAILABLE:

    from pyspark.sql import SparkSession

    spark = (
        SparkSession
        .builder
        .appName(
            "EnergyDemandBlock12"
        )
        .master(
            "local[*]"
        )
        .config(
            "spark.sql.shuffle.partitions",
            "8"
        )
        .config(
            "spark.driver.memory",
            "2g"
        )
        .getOrCreate()
    )


    print(
        "Spark version :",
        spark.version
    )

    print(
        "Spark master :",
        spark.sparkContext.master
    )

Spark version : 4.2.0
Spark master : local[*]


In [143]:
# ==========================================
# SPARK SMOKE TEST
# ==========================================

if PYSPARK_AVAILABLE:

    test_df = spark.createDataFrame(
        [
            (1, "A"),
            (2, "B"),
            (3, "C"),
        ],
        [
            "id",
            "value",
        ],
    )


    test_df.show()

+---+-----+
| id|value|
+---+-----+
|  1|    A|
|  2|    B|
|  3|    C|
+---+-----+



In [144]:
if PYSPARK_AVAILABLE:

    print(
        "Partitions du test :",
        test_df.rdd.getNumPartitions()
    )

Partitions du test : 12


### Lazy Evaluation

Spark utilise principalement une stratégie d'évaluation paresseuse.

Une transformation comme :

```python
df.filter(...)

In [145]:
# ==========================================
# SAVE BLOCK 12 CONFIGURATION
# ==========================================

BLOCK12_CONFIG_PATH = (
    MODEL_DIR
    / "block12_config.joblib"
)


joblib.dump(
    big_data_config,
    BLOCK12_CONFIG_PATH,
)


print(
    "Sauvegardé :",
    BLOCK12_CONFIG_PATH
)

Sauvegardé : C:\Users\thiba\OneDrive\Bureau\Projet energie\models\block12_config.joblib


In [146]:
# ==========================================
# STEP 1 VALIDATION
# ==========================================

assert (
    USE_SPARK_FOR_PRODUCTION
    is False
)

assert (
    RUN_SPARK_LAB
    is True
)

assert (
    KEEP_POSTGRESQL_PIPELINE
    is True
)

assert (
    big_data_config[
        "uses_2026"
    ]
    is False
)


print(
    "Étape 1 : OK"
)

Étape 1 : OK


### Partie C — Analyse et améliorations guidées par la théorie

Un moteur distribué n'est pas automatiquement préférable à un
traitement local.

Pour un dataset de taille modérée, le coût de création des partitions,
de sérialisation des données et de communication peut rendre Spark plus
lent qu'un traitement Pandas classique.

L'intérêt de Spark apparaît principalement lorsque la taille du
problème augmente suffisamment pour exploiter le parallélisme et la
distribution.

Spark représente un DataFrame comme plusieurs partitions :

$$
D =
\bigcup_{j=1}^{P}D_j.
$$

Les transformations peuvent alors être appliquées en parallèle sur
plusieurs partitions.

Spark utilise également une évaluation paresseuse :

$$
\text{transformations}
\rightarrow
\text{plan logique}
\rightarrow
\text{optimisation}
\rightarrow
\text{plan physique}
\rightarrow
\text{action}.
$$

Dans ce projet, Spark est donc utilisé comme laboratoire de montée en
charge et non comme remplacement artificiel du pipeline PostgreSQL
existant.

Conclusion :

L'architecture Big Data du projet est maintenant définie.

PostgreSQL reste la solution utilisée dans le pipeline principal,
tandis que Spark est introduit comme environnement expérimental de
calcul distribué.

Cette séparation permettra d'étudier la montée en charge, les
partitions, les transformations distribuées et les pipelines ML sans
complexifier inutilement le système actuellement en production.

L'année 2026 reste réservée au test final.

## Étape 2 — Chargement du dataset énergétique dans Spark

### Partie A — Mise en pratique

Le dataset de Machine Learning construit dans PostgreSQL est maintenant
chargé dans Spark afin d'étudier son traitement distribué.

Pour cette première intégration, l'architecture utilisée est :

$$
\text{PostgreSQL}
\rightarrow
\text{Pandas}
\rightarrow
\text{Spark}.
$$

Cette solution permet de réutiliser la couche d'accès PostgreSQL déjà
développée dans le Bloc 8.

L'année 2026 reste exclue du chargement.

In [147]:
# ==========================================
# POSTGRESQL DATA ACCESS
# ==========================================

import sys


SRC_DIR = (
    PROJECT_DIR
    / "src"
)


if str(PROJECT_DIR) not in sys.path:

    sys.path.append(
        str(PROJECT_DIR)
    )


from src.database.postgres import (
    load_ml_dataset,
)


print(
    "PostgreSQL loader : OK"
)

PostgreSQL loader : OK


In [148]:
# ==========================================
# POSTGRESQL PASSWORD
# ==========================================

import getpass


if not os.environ.get(
    "POSTGRES_PASSWORD"
):

    os.environ[
        "POSTGRES_PASSWORD"
    ] = getpass.getpass(
        "Mot de passe PostgreSQL : "
    )


print(
    "POSTGRES_PASSWORD configuré :",
    bool(
        os.environ.get(
            "POSTGRES_PASSWORD"
        )
    )
)

POSTGRES_PASSWORD configuré : True


In [149]:
# ==========================================
# DATA PERIOD
# ==========================================

DATA_START = (
    "2013-01-14"
)

DATA_END = (
    "2025-12-31 23:00:00+00"
)


print(
    "Début :",
    DATA_START
)

print(
    "Fin :",
    DATA_END
)

print(
    "2026 exclu : True"
)

Début : 2013-01-14
Fin : 2025-12-31 23:00:00+00
2026 exclu : True


In [150]:
# ==========================================
# LOAD FROM POSTGRESQL
# ==========================================

load_start = (
    time.perf_counter()
)


energy_pd = load_ml_dataset(
    start=DATA_START,
    end=DATA_END,
)


postgres_load_seconds = (
    time.perf_counter()
    - load_start
)


print(
    "Shape Pandas :",
    energy_pd.shape
)

print(
    "Temps PostgreSQL -> Pandas :",
    f"{postgres_load_seconds:.3f} s"
)

Shape Pandas : (227231, 23)
Temps PostgreSQL -> Pandas : 4.020 s


In [151]:
# ==========================================
# PANDAS DATA CHECK
# ==========================================

print(
    "Colonnes :"
)


for column in energy_pd.columns:

    print(
        "-",
        column
    )

Colonnes :
- timestamp_utc
- consumption_mw
- consumption_lag_30m
- consumption_lag_1h
- consumption_lag_24h
- consumption_lag_7d
- consumption_mean_3h
- consumption_mean_24h
- consumption_mean_7d
- consumption_std_24h
- temperature
- temp_lag_1h
- temp_lag_24h
- temp_lag_7d
- temp_mean_24h
- local_hour
- day_of_week
- month
- is_weekend
- hour_sin
- hour_cos
- month_sin
- month_cos


In [152]:
display(
    energy_pd.head()
)

,timestamp_utc,consumption_mw,consumption_lag_30m,consumption_lag_1h,consumption_lag_24h,consumption_lag_7d,consumption_mean_3h,consumption_mean_24h,consumption_mean_7d,consumption_std_24h,...,temp_lag_7d,temp_mean_24h,local_hour,day_of_week,month,is_weekend,hour_sin,hour_cos,month_sin,month_cos
0,2013-01-14 23:00:00+00:00,76172.0,76385.0,77776.0,69414.0,67639.0,76885.666667,76181.916667,70405.812500,7589.262678,...,5.94,1.943333,0E-20,2,1,0,0.000000,1.000000,0.5,0.866025
1,2013-01-14 23:30:00+00:00,74149.0,76172.0,76385.0,67983.0,65881.0,76349.666667,76322.708333,70431.208333,7523.436584,...,5.94,1.912917,0.50000000000000000000,2,1,0,0.130526,0.991445,0.5,0.866025
2,2013-01-15 00:00:00+00:00,71358.0,74149.0,76172.0,65653.0,62950.0,75874.333333,76451.166667,70455.815476,7430.072315,...,5.64,1.882500,1.00000000000000000000,2,1,0,0.258819,0.965926,0.5,0.866025
3,2013-01-15 00:30:00+00:00,71331.0,71358.0,74149.0,65677.0,62842.0,75237.500000,76570.020833,70480.839286,7298.123756,...,5.64,1.864167,1.50000000000000000000,2,1,0,0.382683,0.923880,0.5,0.866025
4,2013-01-15 01:00:00+00:00,71098.0,71331.0,71358.0,65441.0,62289.0,74528.500000,76687.812500,70506.104167,7162.945622,...,5.34,1.845833,2.00000000000000000000,2,1,0,0.500000,0.866025,0.5,0.866025


In [153]:
# ==========================================
# TIMESTAMP COLUMN DETECTION
# ==========================================

timestamp_candidates = [

    column
    for column in energy_pd.columns
    if (
        "timestamp"
        in column.lower()
        or "date"
        in column.lower()
    )
]


print(
    "Colonnes temporelles possibles :",
    timestamp_candidates
)

Colonnes temporelles possibles : ['timestamp_utc']


In [154]:
TIMESTAMP_COLUMN = (
    "timestamp_utc"
)

In [155]:
energy_pd[
    TIMESTAMP_COLUMN
] = pd.to_datetime(
    energy_pd[
        TIMESTAMP_COLUMN
    ],
    utc=True,
)

In [156]:
print(
    "Min :",
    energy_pd[
        TIMESTAMP_COLUMN
    ].min()
)

print(
    "Max :",
    energy_pd[
        TIMESTAMP_COLUMN
    ].max()
)

Min : 2013-01-14 23:00:00+00:00
Max : 2025-12-31 22:00:00+00:00


In [157]:
assert (
    energy_pd[
        TIMESTAMP_COLUMN
    ].dt.year.max()
    <= 2025
)


print(
    "2026 absent : OK"
)

2026 absent : OK


In [158]:
# ==========================================
# PANDAS -> SPARK
# ==========================================

spark_conversion_start = (
    time.perf_counter()
)


energy_spark = (
    spark.createDataFrame(
        energy_pd
    )
)


spark_conversion_seconds = (
    time.perf_counter()
    - spark_conversion_start
)


print(
    "Conversion Pandas -> Spark :",
    f"{spark_conversion_seconds:.3f} s"
)

Conversion Pandas -> Spark : 2.818 s


In [159]:
# ==========================================
# SPARK COUNT ACTION
# ==========================================

count_start = (
    time.perf_counter()
)


spark_row_count = (
    energy_spark.count()
)


spark_count_seconds = (
    time.perf_counter()
    - count_start
)


print(
    "Nombre de lignes Spark :",
    spark_row_count
)

print(
    "Temps count() :",
    f"{spark_count_seconds:.3f} s"
)

Nombre de lignes Spark : 227231
Temps count() : 0.965 s


In [160]:
assert (
    spark_row_count
    == len(
        energy_pd
    )
)


print(
    "Pandas / Spark row count : OK"
)

Pandas / Spark row count : OK


In [161]:
# ==========================================
# SPARK SCHEMA
# ==========================================

energy_spark.printSchema()

root
 |-- timestamp_utc: timestamp (nullable = true)
 |-- consumption_mw: double (nullable = true)
 |-- consumption_lag_30m: double (nullable = true)
 |-- consumption_lag_1h: double (nullable = true)
 |-- consumption_lag_24h: double (nullable = true)
 |-- consumption_lag_7d: double (nullable = true)
 |-- consumption_mean_3h: double (nullable = true)
 |-- consumption_mean_24h: double (nullable = true)
 |-- consumption_mean_7d: double (nullable = true)
 |-- consumption_std_24h: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- temp_lag_1h: double (nullable = true)
 |-- temp_lag_24h: double (nullable = true)
 |-- temp_lag_7d: double (nullable = true)
 |-- temp_mean_24h: double (nullable = true)
 |-- local_hour: decimal(22,20) (nullable = true)
 |-- day_of_week: decimal(1,0) (nullable = true)
 |-- month: decimal(2,0) (nullable = true)
 |-- is_weekend: long (nullable = true)
 |-- hour_sin: double (nullable = true)
 |-- hour_cos: double (nullable = true)
 |-- month_sin

In [162]:
# ==========================================
# SPARK PREVIEW
# ==========================================

energy_spark.show(
    n=5,
    truncate=False,
)

+-------------------+--------------+-------------------+------------------+-------------------+------------------+-------------------+--------------------+-------------------+-------------------+------------------+-----------+------------+-----------------+------------------+----------------------+-----------+-----+----------+-------------------+------------------+-------------------+------------------+
|timestamp_utc      |consumption_mw|consumption_lag_30m|consumption_lag_1h|consumption_lag_24h|consumption_lag_7d|consumption_mean_3h|consumption_mean_24h|consumption_mean_7d|consumption_std_24h|temperature       |temp_lag_1h|temp_lag_24h|temp_lag_7d      |temp_mean_24h     |local_hour            |day_of_week|month|is_weekend|hour_sin           |hour_cos          |month_sin          |month_cos         |
+-------------------+--------------+-------------------+------------------+-------------------+------------------+-------------------+--------------------+-------------------+-----------

In [163]:
energy_spark.select(
    TIMESTAMP_COLUMN,
    "consumption_mw",
).show(
    5,
    truncate=False,
)

+-------------------+--------------+
|timestamp_utc      |consumption_mw|
+-------------------+--------------+
|2013-01-14 23:00:00|76172.0       |
|2013-01-14 23:30:00|74149.0       |
|2013-01-15 00:00:00|71358.0       |
|2013-01-15 00:30:00|71331.0       |
|2013-01-15 01:00:00|71098.0       |
+-------------------+--------------+
only showing top 5 rows


In [164]:
# ==========================================
# DATASET DIMENSIONS
# ==========================================

spark_column_count = len(
    energy_spark.columns
)


print(
    "Lignes :",
    spark_row_count
)

print(
    "Colonnes :",
    spark_column_count
)

Lignes : 227231
Colonnes : 23


In [165]:
# ==========================================
# CURRENT PARTITIONS
# ==========================================

initial_partitions = (
    energy_spark
    .rdd
    .getNumPartitions()
)


print(
    "Partitions initiales :",
    initial_partitions
)

Partitions initiales : 12


In [166]:
# ==========================================
# REPARTITION
# ==========================================

N_PARTITIONS = 8


energy_spark_partitioned = (
    energy_spark
    .repartition(
        N_PARTITIONS
    )
)


print(
    "Partitions après repartition :",
    energy_spark_partitioned
    .rdd
    .getNumPartitions()
)

Partitions après repartition : 8


### Repartitionnement

La méthode :

```python
df.repartition(8)

redistribue les observations entre huit partitions.

Chaque partition peut être traitée indépendamment.
Cependant, le repartitionnement nécessite généralement un échange de
données entre partitions appelé :

Un nombre trop faible de partitions limite le parallélisme.
Un nombre trop élevé augmente les coûts de planification et de
communication.
Le nombre optimal dépend donc :
- du volume des données 
- du nombre de cœurs 
- de la mémoire 
- du type d'opération.

In [167]:
# ==========================================
# PARTITION SIZES
# ==========================================

partition_sizes = (
    energy_spark_partitioned
    .rdd
    .mapPartitions(
        lambda rows: [
            sum(
                1
                for _ in rows
            )
        ]
    )
    .collect()
)


partition_distribution = pd.DataFrame({

    "partition":
        range(
            len(
                partition_sizes
            )
        ),

    "rows":
        partition_sizes,
})


display(
    partition_distribution
)

,partition,rows
0,0,28404
1,1,28403
2,2,28404
3,3,28404
4,4,28404
5,5,28404
6,6,28404
7,7,28404


In [168]:
print(
    "Minimum :",
    min(
        partition_sizes
    )
)

print(
    "Maximum :",
    max(
        partition_sizes
    )
)

print(
    "Moyenne :",
    np.mean(
        partition_sizes
    )
)

Minimum : 28403
Maximum : 28404
Moyenne : 28403.875


In [169]:
# ==========================================
# PARTITION IMBALANCE
# ==========================================

partition_mean = np.mean(
    partition_sizes
)


partition_imbalance = (
    max(
        partition_sizes
    )
    / partition_mean
)


print(
    "Ratio max / moyenne :",
    f"{partition_imbalance:.3f}"
)

Ratio max / moyenne : 1.000


In [170]:
# ==========================================
# CACHE DATASET
# ==========================================

energy_spark_cached = (
    energy_spark_partitioned
    .cache()
)

In [171]:
cache_start = (
    time.perf_counter()
)


cached_count = (
    energy_spark_cached.count()
)


cache_materialization_seconds = (
    time.perf_counter()
    - cache_start
)


print(
    "Rows cached :",
    cached_count
)

print(
    "Matérialisation cache :",
    f"{cache_materialization_seconds:.3f} s"
)

Rows cached : 227231
Matérialisation cache : 0.326 s


In [172]:
second_count_start = (
    time.perf_counter()
)


second_count = (
    energy_spark_cached.count()
)


second_count_seconds = (
    time.perf_counter()
    - second_count_start
)


print(
    "Second count :",
    second_count
)

print(
    "Temps avec cache :",
    f"{second_count_seconds:.3f} s"
)

Second count : 227231
Temps avec cache : 0.268 s


In [173]:
# ==========================================
# STEP 2 PERFORMANCE
# ==========================================

step2_performance = pd.DataFrame({

    "Operation": [
        "PostgreSQL -> Pandas",
        "Pandas -> Spark",
        "First Spark count",
        "Cache materialization",
        "Cached count",
    ],

    "Seconds": [
        postgres_load_seconds,
        spark_conversion_seconds,
        spark_count_seconds,
        cache_materialization_seconds,
        second_count_seconds,
    ],
})


display(
    step2_performance
)

,Operation,Seconds
0,PostgreSQL -> Pandas,4.019542
1,Pandas -> Spark,2.817793
2,First Spark count,0.965107
3,Cache materialization,0.325602
4,Cached count,0.267910


In [174]:
# ==========================================
# SAVE STEP 2 METADATA
# ==========================================

STEP2_PATH = (
    BLOCK12_DIR
    / "step2_metadata.joblib"
)


step2_state = {

    "rows":
        spark_row_count,

    "columns":
        spark_column_count,

    "partitions":
        N_PARTITIONS,

    "partition_sizes":
        partition_sizes,

    "partition_imbalance":
        partition_imbalance,

    "performance":
        step2_performance,

    "spark_version":
        spark.version,

    "uses_2026":
        False,
}


joblib.dump(
    step2_state,
    STEP2_PATH
)


print(
    "Métadonnées sauvegardées :",
    STEP2_PATH
)

Métadonnées sauvegardées : C:\Users\thiba\OneDrive\Bureau\Projet energie\data_processed\block12\step2_metadata.joblib


In [175]:
# ==========================================
# STEP 2 VALIDATION
# ==========================================

assert (
    spark_row_count
    == len(
        energy_pd
    )
)


assert (
    spark_row_count
    > 0
)


assert (
    spark_column_count
    > 0
)


assert (
    energy_pd[
        TIMESTAMP_COLUMN
    ].dt.year.max()
    <= 2025
)


assert (
    sum(
        partition_sizes
    )
    == spark_row_count
)


assert (
    len(
        partition_sizes
    )
    == N_PARTITIONS
)


print(
    "Étape 2 : OK"
)

Étape 2 : OK


### Partie C — Analyse et améliorations guidées par la théorie

Le dataset énergétique réel est maintenant représenté sous forme de
DataFrame Spark.

La conversion permet de passer d'un traitement local en mémoire à une
représentation partitionnée :

$$
D =
\bigcup_{j=1}^{P} D_j.
$$

Le repartitionnement permet de distribuer approximativement les
observations entre plusieurs partitions.

Cependant, `repartition` peut provoquer un shuffle, c'est-à-dire un
transfert de données entre partitions.

Ce coût n'est intéressant que si les traitements ultérieurs sont
suffisamment importants pour bénéficier du parallélisme.

Le mécanisme de cache permet quant à lui de conserver en mémoire les
résultats d'un DataFrame après sa première matérialisation.

Une première action :

$$
count(D)
$$

déclenche le calcul et la mise en cache.

Les actions suivantes peuvent ensuite réutiliser ces données sans
recalculer l'ensemble du plan.

Pour le volume actuel du projet, Spark n'est pas nécessairement plus
rapide que Pandas ou PostgreSQL.

Cette expérience fournit cependant la base nécessaire pour étudier la
montée en charge vers plusieurs millions d'observations.

Conclusion :

Le dataset énergétique réel a été chargé avec succès dans Spark tout en
maintenant l'exclusion complète de l'année 2026.

Les données ont été réparties sur plusieurs partitions et mises en cache
afin de préparer les traitements distribués des prochaines étapes.

Les temps de chargement, de matérialisation et de lecture depuis le
cache sont enregistrés afin de pouvoir comparer objectivement Spark,
Pandas et PostgreSQL.

Le pipeline principal reste basé sur PostgreSQL ; Spark reste pour
l'instant une branche expérimentale de montée en charge.

## Étape 3 — Feature Engineering distribué avec Spark

### Partie A — Mise en pratique

Cette étape reconstruit plusieurs variables temporelles directement
dans Spark.

L'objectif est d'étudier comment les opérations réalisées auparavant
avec Pandas ou SQL peuvent être exprimées sous forme de transformations
distribuées.

On construit notamment :

- les variables calendaires ;
- les lags temporels ;
- les moyennes mobiles ;
- les différences temporelles ;
- les statistiques horaires et journalières.

L'année 2026 reste exclue.

In [176]:
# ==========================================
# COLUMN INSPECTION
# ==========================================

print(
    "Colonnes Spark :"
)

for column in energy_spark_cached.columns:

    print(
        "-",
        column
    )

Colonnes Spark :
- timestamp_utc
- consumption_mw
- consumption_lag_30m
- consumption_lag_1h
- consumption_lag_24h
- consumption_lag_7d
- consumption_mean_3h
- consumption_mean_24h
- consumption_mean_7d
- consumption_std_24h
- temperature
- temp_lag_1h
- temp_lag_24h
- temp_lag_7d
- temp_mean_24h
- local_hour
- day_of_week
- month
- is_weekend
- hour_sin
- hour_cos
- month_sin
- month_cos


In [177]:
# ==========================================
# TARGET COLUMN DETECTION
# ==========================================

TARGET_CANDIDATES = [
    "consumption_mw",
    "consumption",
    "actual_mw",
    "value",
]


TARGET_COLUMN = next(
    (
        column
        for column in TARGET_CANDIDATES
        if column
        in energy_spark_cached.columns
    ),
    None,
)


print(
    "Timestamp :",
    TIMESTAMP_COLUMN
)

print(
    "Target détectée :",
    TARGET_COLUMN
)

Timestamp : timestamp_utc
Target détectée : consumption_mw


In [178]:
if TARGET_COLUMN is None:

    raise ValueError(
        "Impossible de détecter automatiquement "
        "la colonne de consommation. "
        "Regarde energy_spark_cached.columns "
        "et définis TARGET_COLUMN manuellement."
    )

In [179]:
# ==========================================
# SPARK SQL FUNCTIONS
# ==========================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [180]:
# ==========================================
# TIMESTAMP CHECK
# ==========================================

energy_spark_cached.select(
    TIMESTAMP_COLUMN
).printSchema()

root
 |-- timestamp_utc: timestamp (nullable = true)



In [181]:
energy_features = (
    energy_spark_cached
    .withColumn(
        TIMESTAMP_COLUMN,
        F.to_timestamp(
            F.col(
                TIMESTAMP_COLUMN
            )
        )
    )
)

In [182]:
# ==========================================
# CALENDAR FEATURES
# ==========================================

energy_features = (
    energy_features

    .withColumn(
        "hour",
        F.hour(
            F.col(
                TIMESTAMP_COLUMN
            )
        )
    )

    .withColumn(
        "day_of_month",
        F.dayofmonth(
            F.col(
                TIMESTAMP_COLUMN
            )
        )
    )

    .withColumn(
        "month",
        F.month(
            F.col(
                TIMESTAMP_COLUMN
            )
        )
    )

    .withColumn(
        "year",
        F.year(
            F.col(
                TIMESTAMP_COLUMN
            )
        )
    )

    .withColumn(
        "day_of_week",
        F.dayofweek(
            F.col(
                TIMESTAMP_COLUMN
            )
        )
    )
)

In [183]:
energy_features = (
    energy_features
    .withColumn(
        "is_weekend",
        F.when(
            F.col(
                "day_of_week"
            ).isin(
                1,
                7,
            ),
            1,
        )
        .otherwise(
            0
        )
    )
)

In [184]:
# ==========================================
# CYCLICAL TIME FEATURES
# ==========================================

energy_features = (
    energy_features

    .withColumn(
        "hour_sin",
        F.sin(
            2
            * np.pi
            * F.col(
                "hour"
            )
            / 24
        )
    )

    .withColumn(
        "hour_cos",
        F.cos(
            2
            * np.pi
            * F.col(
                "hour"
            )
            / 24
        )
    )

    .withColumn(
        "month_sin",
        F.sin(
            2
            * np.pi
            * F.col(
                "month"
            )
            / 12
        )
    )

    .withColumn(
        "month_cos",
        F.cos(
            2
            * np.pi
            * F.col(
                "month"
            )
            / 12
        )
    )
)

In [185]:
# ==========================================
# TEMPORAL WINDOW
# ==========================================

time_window = (
    Window
    .orderBy(
        F.col(
            TIMESTAMP_COLUMN
        )
    )
)

In [186]:
# ==========================================
# LAG FEATURES
# ==========================================

LAG_STEPS = {

    "lag_1h":
        2,

    "lag_24h":
        48,

    "lag_7d":
        336,
}


for feature_name, lag_steps in (
    LAG_STEPS.items()
):

    energy_features = (
        energy_features
        .withColumn(
            feature_name,
            F.lag(
                F.col(
                    TARGET_COLUMN
                ),
                lag_steps,
            )
            .over(
                time_window
            )
        )
    )

In [187]:
# ==========================================
# TEMPORAL DIFFERENCE
# ==========================================

energy_features = (
    energy_features
    .withColumn(
        "diff_24h",
        F.col(
            TARGET_COLUMN
        )
        - F.col(
            "lag_24h"
        )
    )
)

In [188]:
rolling_24h_window = (
    time_window
    .rowsBetween(
        -48,
        -1,
    )
)

In [189]:
# ==========================================
# ROLLING 24H FEATURES
# ==========================================

energy_features = (
    energy_features

    .withColumn(
        "rolling_mean_24h",
        F.avg(
            F.col(
                TARGET_COLUMN
            )
        )
        .over(
            rolling_24h_window
        )
    )

    .withColumn(
        "rolling_std_24h",
        F.stddev(
            F.col(
                TARGET_COLUMN
            )
        )
        .over(
            rolling_24h_window
        )
    )
)

In [190]:
# ==========================================
# ROLLING 7D FEATURE
# ==========================================

rolling_7d_window = (
    time_window
    .rowsBetween(
        -336,
        -1,
    )
)


energy_features = (
    energy_features
    .withColumn(
        "rolling_mean_7d",
        F.avg(
            F.col(
                TARGET_COLUMN
            )
        )
        .over(
            rolling_7d_window
        )
    )
)

In [191]:
# ==========================================
# MATERIALIZE FEATURE ENGINEERING
# ==========================================

feature_start = (
    time.perf_counter()
)


feature_row_count = (
    energy_features.count()
)


feature_engineering_seconds = (
    time.perf_counter()
    - feature_start
)


print(
    "Lignes :",
    feature_row_count
)

print(
    "Temps feature engineering :",
    f"{feature_engineering_seconds:.3f} s"
)

Lignes : 227231
Temps feature engineering : 0.305 s


In [192]:
# ==========================================
# FEATURE PREVIEW
# ==========================================

energy_features.select(

    TIMESTAMP_COLUMN,

    TARGET_COLUMN,

    "hour",

    "is_weekend",

    "lag_1h",

    "lag_24h",

    "lag_7d",

    "rolling_mean_24h",

    "rolling_mean_7d",

).show(
    10,
    truncate=False,
)

+-------------------+--------------+----+----------+-------+-------+------+-----------------+-----------------+
|timestamp_utc      |consumption_mw|hour|is_weekend|lag_1h |lag_24h|lag_7d|rolling_mean_24h |rolling_mean_7d  |
+-------------------+--------------+----+----------+-------+-------+------+-----------------+-----------------+
|2013-01-14 23:00:00|76172.0       |23  |0         |NULL   |NULL   |NULL  |NULL             |NULL             |
|2013-01-14 23:30:00|74149.0       |23  |0         |NULL   |NULL   |NULL  |76172.0          |76172.0          |
|2013-01-15 00:00:00|71358.0       |0   |0         |76172.0|NULL   |NULL  |75160.5          |75160.5          |
|2013-01-15 00:30:00|71331.0       |0   |0         |74149.0|NULL   |NULL  |73893.0          |73893.0          |
|2013-01-15 01:00:00|71098.0       |1   |0         |71358.0|NULL   |NULL  |73252.5          |73252.5          |
|2013-01-15 01:30:00|70688.0       |1   |0         |71331.0|NULL   |NULL  |72821.6          |72821.6    

In [193]:
# ==========================================
# NULL ANALYSIS
# ==========================================

FEATURE_COLUMNS = [

    "lag_1h",

    "lag_24h",

    "lag_7d",

    "rolling_mean_24h",

    "rolling_std_24h",

    "rolling_mean_7d",
]


null_counts_expr = [

    F.sum(
        F.col(
            column
        ).isNull().cast(
            "int"
        )
    ).alias(
        column
    )

    for column
    in FEATURE_COLUMNS
]


null_counts = (
    energy_features
    .select(
        null_counts_expr
    )
)


null_counts.show(
    truncate=False
)

+------+-------+------+----------------+---------------+---------------+
|lag_1h|lag_24h|lag_7d|rolling_mean_24h|rolling_std_24h|rolling_mean_7d|
+------+-------+------+----------------+---------------+---------------+
|2     |48     |336   |1               |2              |1              |
+------+-------+------+----------------+---------------+---------------+



In [194]:
# ==========================================
# CLEAN FEATURE DATASET
# ==========================================

energy_features_clean = (
    energy_features
    .dropna(
        subset=FEATURE_COLUMNS
    )
)

In [195]:
clean_count = (
    energy_features_clean.count()
)


print(
    "Avant :",
    feature_row_count
)

print(
    "Après dropna :",
    clean_count
)

print(
    "Lignes supprimées :",
    (
        feature_row_count
        - clean_count
    )
)

Avant : 227231
Après dropna : 226895
Lignes supprimées : 336


In [196]:
# ==========================================
# FINAL YEAR CHECK
# ==========================================

max_year = (
    energy_features_clean
    .agg(
        F.max(
            "year"
        )
        .alias(
            "max_year"
        )
    )
    .collect()[0][
        "max_year"
    ]
)


print(
    "Année maximale :",
    max_year
)


assert (
    max_year
    <= 2025
)

Année maximale : 2025


In [197]:
# ==========================================
# HOURLY AGGREGATION
# ==========================================

hourly_profile = (
    energy_features_clean

    .groupBy(
        "hour"
    )

    .agg(

        F.avg(
            TARGET_COLUMN
        ).alias(
            "mean_consumption"
        ),

        F.stddev(
            TARGET_COLUMN
        ).alias(
            "std_consumption"
        ),

        F.count(
            "*"
        ).alias(
            "n"
        ),
    )

    .orderBy(
        "hour"
    )
)

In [198]:
hourly_profile.show(
    24,
    truncate=False,
)

+----+------------------+------------------+----+
|hour|mean_consumption  |std_consumption   |n   |
+----+------------------+------------------+----+
|0   |48172.839750370214|10575.397348035835|9454|
|1   |46511.107414850856|11211.04530453044 |9454|
|2   |44766.527924687965|10738.66017179115 |9454|
|3   |44410.88629151682 |10007.352382190795|9454|
|4   |46037.460334250056|9885.274646976013 |9454|
|5   |49395.48244129469 |10773.311175511235|9454|
|6   |52979.6220647345  |11972.999988282636|9454|
|7   |55410.97144066004 |12098.039268151899|9454|
|8   |56721.43759255342 |11674.710433967419|9454|
|9   |57362.665538396446|11174.154040864956|9454|
|10  |57968.20277131373 |10456.692241208251|9454|
|11  |58059.558387983925|10349.038253948293|9454|
|12  |56859.45039136873 |10647.127562397889|9454|
|13  |55115.13814258515 |10418.12236728607 |9454|
|14  |53485.75745716099 |10135.00029291363 |9454|
|15  |52477.93082293209 |10008.435199097534|9454|
|16  |53064.59794795854 |10149.271507156103|9454|


In [199]:
# ==========================================
# WEEKEND AGGREGATION
# ==========================================

weekend_profile = (
    energy_features_clean

    .groupBy(
        "is_weekend"
    )

    .agg(

        F.avg(
            TARGET_COLUMN
        ).alias(
            "mean_consumption"
        ),

        F.stddev(
            TARGET_COLUMN
        ).alias(
            "std_consumption"
        ),

        F.count(
            "*"
        ).alias(
            "n"
        ),
    )

    .orderBy(
        "is_weekend"
    )
)


weekend_profile.show()

+----------+------------------+------------------+------+
|is_weekend|  mean_consumption|   std_consumption|     n|
+----------+------------------+------------------+------+
|         0|54478.297381165365|11473.244514225835|162095|
|         1| 48359.60360339506| 10604.15033111562| 64800|
+----------+------------------+------------------+------+



In [200]:
# ==========================================
# MONTHLY AGGREGATION
# ==========================================

monthly_profile = (
    energy_features_clean

    .groupBy(
        "month"
    )

    .agg(

        F.avg(
            TARGET_COLUMN
        ).alias(
            "mean_consumption"
        ),

        F.count(
            "*"
        ).alias(
            "n"
        ),
    )

    .orderBy(
        "month"
    )
)


monthly_profile.show(
    12
)

+-----+------------------+-----+
|month|  mean_consumption|    n|
+-----+------------------+-----+
|    1| 68401.96766277675|18338|
|    2| 65641.80091961853|17616|
|    3| 58619.13262510339|19344|
|    4| 50207.94241452991|18720|
|    5| 45051.30536600496|19344|
|    6| 44460.80977564103|18720|
|    7| 44952.32190860215|19344|
|    8|  42068.3163254756|19344|
|    9| 44817.58322649573|18720|
|   10| 48433.57578577337|19344|
|   11|57877.688408119655|18720|
|   12|63770.171397549246|19341|
+-----+------------------+-----+



In [201]:
# ==========================================
# SPARK EXECUTION PLAN
# ==========================================

hourly_profile.explain(
    mode="formatted"
)

== Physical Plan ==
AdaptiveSparkPlan (17)
+- Sort (16)
   +- HashAggregate (15)
      +- HashAggregate (14)
         +- Project (13)
            +- Filter (12)
               +- Window (11)
                  +- Sort (10)
                     +- Exchange (9)
                        +- Project (8)
                           +- InMemoryTableScan (1)
                                 +- InMemoryRelation (2)
                                       +- AdaptiveSparkPlan (7)
                                          +- == Final Plan ==
                                             ResultQueryStage (6)
                                             +- ShuffleQueryStage (5), Statistics(sizeInBytes=45.1 MiB, rowCount=2.27E+5)
                                                +- Exchange (4)
                                                   +- LocalTableScan (3)
                                          +- == Initial Plan ==
                                             Exchange (4)
                    

In [202]:
# ==========================================
# CACHE FEATURE DATASET
# ==========================================

energy_features_cached = (
    energy_features_clean
    .cache()
)

In [203]:
feature_cache_start = (
    time.perf_counter()
)


feature_cache_count = (
    energy_features_cached.count()
)


feature_cache_seconds = (
    time.perf_counter()
    - feature_cache_start
)


print(
    "Feature cache :",
    f"{feature_cache_seconds:.3f} s"
)

Feature cache : 5.120 s


In [204]:
cached_feature_start = (
    time.perf_counter()
)


cached_feature_count = (
    energy_features_cached.count()
)


cached_feature_seconds = (
    time.perf_counter()
    - cached_feature_start
)


print(
    "Count après cache :",
    f"{cached_feature_seconds:.3f} s"
)

Count après cache : 0.278 s


In [205]:
# ==========================================
# STEP 3 PERFORMANCE SUMMARY
# ==========================================

step3_performance = pd.DataFrame({

    "Operation": [
        "Feature engineering",
        "Feature cache materialization",
        "Cached feature count",
    ],

    "Seconds": [
        feature_engineering_seconds,
        feature_cache_seconds,
        cached_feature_seconds,
    ],
})


display(
    step3_performance
)

,Operation,Seconds
0,Feature engineering,0.305434
1,Feature cache materialization,5.119769
2,Cached feature count,0.278118


In [206]:
# ==========================================
# SAVE STEP 3
# ==========================================

STEP3_PATH = (
    BLOCK12_DIR
    / "step3_features_metadata.joblib"
)


step3_state = {

    "rows_before_cleaning":
        feature_row_count,

    "rows_after_cleaning":
        clean_count,

    "feature_columns":
        FEATURE_COLUMNS,

    "performance":
        step3_performance,

    "max_year":
        max_year,

    "uses_2026":
        False,
}


joblib.dump(
    step3_state,
    STEP3_PATH
)


print(
    "Sauvegardé :",
    STEP3_PATH
)

Sauvegardé : C:\Users\thiba\OneDrive\Bureau\Projet energie\data_processed\block12\step3_features_metadata.joblib


### Partie C — Analyse et améliorations guidées par la théorie

Le feature engineering temporel peut être exprimé dans Spark à l'aide
de fonctions de fenêtre.

Un lag de longueur $h$ correspond à :

$$
x_{t-h}.
$$

À fréquence 30 minutes :

$$
h_{1h}=2,
$$

$$
h_{24h}=48,
$$

et :

$$
h_{7j}=336.
$$

Les moyennes mobiles sont également construites uniquement à partir du
passé.

Pour une fenêtre de longueur $L$ :

$$
\bar{x}_{t,L}
=
\frac{1}{L}
\sum_{j=1}^{L}x_{t-j}.
$$

La valeur présente $x_t$ est volontairement exclue afin d'éviter toute
fuite d'information.

Spark exécute ces transformations de manière paresseuse.

Les calculs sont réellement matérialisés lorsqu'une action telle que
`count()` ou `show()` est déclenchée.

Les agrégations `groupBy` nécessitent souvent un shuffle entre les
partitions, visible dans le plan physique sous la forme d'opérations
`Exchange`.

Le coût de ces échanges constituera un élément central des prochaines
expériences d'optimisation.

Conclusion :

L'Étape 3 transpose le feature engineering temporel du projet dans un
pipeline Spark.

Les variables calendaires, les lags, les différences temporelles et les
moyennes mobiles sont calculés directement dans le moteur distribué.

Les fenêtres sont construites uniquement à partir d'informations
passées afin d'éviter toute fuite de données.

Le dataset enrichi est mis en cache pour les traitements suivants et
l'année 2026 reste complètement exclue.

In [207]:
# ==========================================
# REAL FEATURE ENGINEERING BENCHMARK
# ==========================================

real_feature_start = (
    time.perf_counter()
)


real_feature_stats = (
    energy_features_clean

    .select(

        F.avg(
            "lag_1h"
        ).alias(
            "avg_lag_1h"
        ),

        F.avg(
            "lag_24h"
        ).alias(
            "avg_lag_24h"
        ),

        F.avg(
            "lag_7d"
        ).alias(
            "avg_lag_7d"
        ),

        F.avg(
            "rolling_mean_24h"
        ).alias(
            "avg_rolling_24h"
        ),

        F.avg(
            "rolling_mean_7d"
        ).alias(
            "avg_rolling_7d"
        ),
    )

    .collect()
)


real_feature_seconds = (
    time.perf_counter()
    - real_feature_start
)


print(
    "Temps réel feature engineering :",
    f"{real_feature_seconds:.3f} s"
)


print(
    real_feature_stats
)

Temps réel feature engineering : 0.369 s
[Row(avg_lag_1h=52730.89517397915, avg_lag_24h=52732.56032305692, avg_lag_7d=52750.348026620246, avg_rolling_24h=52731.82659453393, avg_rolling_7d=52738.8053921113)]


In [208]:
# ==========================================
# WINDOW PARTITION CHECK
# ==========================================

print(
    "Partitions avant fenêtres :",
    energy_spark_cached
    .rdd
    .getNumPartitions()
)

print(
    "Partitions dataset features :",
    energy_features_clean
    .rdd
    .getNumPartitions()
)

Partitions avant fenêtres : 8
Partitions dataset features : 1


In [209]:
energy_features_clean.explain(
    mode="formatted"
)

== Physical Plan ==
AdaptiveSparkPlan (30)
+- == Final Plan ==
   ResultQueryStage (29)
   +- TableCacheQueryStage (28), Statistics(sizeInBytes=51.9 MiB, rowCount=2.27E+5)
      +- InMemoryTableScan (1)
            +- InMemoryRelation (2)
                  +- AdaptiveSparkPlan (27)
                  +- == Final Plan ==
                     ResultQueryStage (19)
                     +- * Project (18)
                        +- * Filter (17)
                           +- Window (16)
                              +- * Sort (15)
                                 +- ShuffleQueryStage (14), Statistics(sizeInBytes=50.3 MiB, rowCount=2.27E+5)
                                    +- Exchange (13)
                                       +- * Project (12)
                                          +- * Project (11)
                                             +- TableCacheQueryStage (10), Statistics(sizeInBytes=39.9 MiB, rowCount=2.27E+5)
                                                +- InMemoryTab

### Limite des fenêtres temporelles globales

Avant le calcul des fenêtres temporelles, le dataset Spark est réparti
sur 8 partitions.

Après l'application d'une fenêtre définie uniquement par :

`ORDER BY timestamp`

le dataset obtenu ne possède plus qu'une seule partition.

On observe donc :

$$
8\ partitions
\rightarrow
1\ partition.
$$

Cette réduction s'explique par la nécessité de maintenir un ordre
temporel global pour calculer correctement les lags et les moyennes
mobiles.

Le plan physique confirme la présence des opérations :

`Exchange -> Sort -> Window`.

Cette architecture limite le parallélisme pour une unique série
temporelle nationale.

Avec plusieurs séries indépendantes, il serait possible d'utiliser :

`partitionBy(series_id)`

afin de calculer les fenêtres temporelles séparément et en parallèle
pour chaque série.

Cette observation constitue une limite importante de l'utilisation de
Spark pour le dataset actuel.

### Partie A — Mise en pratique

## Étape 4 — Spark SQL et agrégations distribuées

Cette étape utilise Spark SQL pour exprimer les mêmes traitements que
ceux réalisés avec l'API DataFrame.

L'objectif est de :

- créer une vue SQL temporaire ;
- réaliser des agrégations horaires, mensuelles et annuelles ;
- utiliser des Window Functions SQL ;
- comparer Spark SQL et l'API DataFrame ;
- analyser les shuffles dans le plan physique ;
- mesurer les performances des agrégations.

Contrairement aux fenêtres temporelles globales de l'étape précédente,
les agrégations par groupe peuvent exploiter plusieurs partitions.

L'année 2026 reste exclue.

In [210]:
# ==========================================
# SPARK SQL VIEW
# ==========================================

energy_features_cached.createOrReplaceTempView(
    "energy_features"
)


print(
    "Vue Spark SQL créée : energy_features"
)

Vue Spark SQL créée : energy_features


In [211]:
# ==========================================
# BASIC SQL CHECK
# ==========================================

basic_sql_check = spark.sql(
    """
    SELECT
        COUNT(*) AS n_rows,
        MIN(year) AS min_year,
        MAX(year) AS max_year
    FROM energy_features
    """
)


basic_sql_check.show()

+------+--------+--------+
|n_rows|min_year|max_year|
+------+--------+--------+
|226895|    2013|    2025|
+------+--------+--------+



In [212]:
sql_check_row = (
    basic_sql_check
    .collect()[0]
)


assert (
    sql_check_row[
        "max_year"
    ]
    <= 2025
)


print(
    "2026 absent : OK"
)

2026 absent : OK


In [213]:
# ==========================================
# HOURLY PROFILE — SPARK SQL
# ==========================================

hourly_sql = spark.sql(
    f"""
    SELECT
        hour,

        AVG(
            {TARGET_COLUMN}
        ) AS mean_consumption,

        STDDEV(
            {TARGET_COLUMN}
        ) AS std_consumption,

        COUNT(*) AS n

    FROM energy_features

    GROUP BY hour

    ORDER BY hour
    """
)


hourly_sql.show(
    24,
    truncate=False,
)

+----+------------------+------------------+----+
|hour|mean_consumption  |std_consumption   |n   |
+----+------------------+------------------+----+
|0   |48172.839750370214|10575.397348035835|9454|
|1   |46511.107414850856|11211.04530453044 |9454|
|2   |44766.527924687965|10738.66017179115 |9454|
|3   |44410.88629151682 |10007.352382190795|9454|
|4   |46037.460334250056|9885.274646976013 |9454|
|5   |49395.48244129469 |10773.311175511235|9454|
|6   |52979.6220647345  |11972.999988282636|9454|
|7   |55410.97144066004 |12098.039268151899|9454|
|8   |56721.43759255342 |11674.710433967419|9454|
|9   |57362.665538396446|11174.154040864956|9454|
|10  |57968.20277131373 |10456.692241208251|9454|
|11  |58059.558387983925|10349.038253948293|9454|
|12  |56859.45039136873 |10647.127562397889|9454|
|13  |55115.13814258515 |10418.12236728607 |9454|
|14  |53485.75745716099 |10135.00029291363 |9454|
|15  |52477.93082293209 |10008.435199097534|9454|
|16  |53064.59794795854 |10149.271507156103|9454|


In [214]:
# ==========================================
# HOURLY PROFILE — SPARK SQL
# ==========================================

hourly_sql = spark.sql(
    f"""
    SELECT
        hour,

        AVG(
            {TARGET_COLUMN}
        ) AS mean_consumption,

        STDDEV(
            {TARGET_COLUMN}
        ) AS std_consumption,

        COUNT(*) AS n

    FROM energy_features

    GROUP BY hour

    ORDER BY hour
    """
)


hourly_sql.show(
    24,
    truncate=False,
)

+----+------------------+------------------+----+
|hour|mean_consumption  |std_consumption   |n   |
+----+------------------+------------------+----+
|0   |48172.839750370214|10575.397348035835|9454|
|1   |46511.107414850856|11211.04530453044 |9454|
|2   |44766.527924687965|10738.66017179115 |9454|
|3   |44410.88629151682 |10007.352382190795|9454|
|4   |46037.460334250056|9885.274646976013 |9454|
|5   |49395.48244129469 |10773.311175511235|9454|
|6   |52979.6220647345  |11972.999988282636|9454|
|7   |55410.97144066004 |12098.039268151899|9454|
|8   |56721.43759255342 |11674.710433967419|9454|
|9   |57362.665538396446|11174.154040864956|9454|
|10  |57968.20277131373 |10456.692241208251|9454|
|11  |58059.558387983925|10349.038253948293|9454|
|12  |56859.45039136873 |10647.127562397889|9454|
|13  |55115.13814258515 |10418.12236728607 |9454|
|14  |53485.75745716099 |10135.00029291363 |9454|
|15  |52477.93082293209 |10008.435199097534|9454|
|16  |53064.59794795854 |10149.271507156103|9454|


In [215]:
# ==========================================
# REBUILD HOURLY PROFILE IF NECESSARY
# ==========================================

if "hourly_profile" not in globals():

    hourly_profile = (
        energy_features_cached

        .groupBy(
            "hour"
        )

        .agg(

            F.avg(
                TARGET_COLUMN
            ).alias(
                "mean_consumption"
            ),

            F.stddev(
                TARGET_COLUMN
            ).alias(
                "std_consumption"
            ),

            F.count(
                "*"
            ).alias(
                "n"
            ),
        )

        .orderBy(
            "hour"
        )
    )

In [216]:
# ==========================================
# SQL VS DATAFRAME API
# ==========================================

hourly_api_pd = (
    hourly_profile
    .toPandas()
    .sort_values(
        "hour"
    )
    .reset_index(
        drop=True
    )
)


hourly_sql_pd = (
    hourly_sql
    .toPandas()
    .sort_values(
        "hour"
    )
    .reset_index(
        drop=True
    )
)

In [217]:
hourly_difference = (
    hourly_api_pd[
        "mean_consumption"
    ]
    - hourly_sql_pd[
        "mean_consumption"
    ]
)


max_hourly_difference = float(
    np.abs(
        hourly_difference
    ).max()
)


print(
    "Différence maximale API / SQL :",
    max_hourly_difference
)

Différence maximale API / SQL : 0.0


In [218]:
assert (
    max_hourly_difference
    < 1e-9
)


print(
    "Spark SQL / DataFrame API : OK"
)

Spark SQL / DataFrame API : OK


In [219]:
# ==========================================
# YEARLY AGGREGATION
# ==========================================

yearly_sql = spark.sql(
    f"""
    SELECT
        year,

        COUNT(*) AS n,

        AVG(
            {TARGET_COLUMN}
        ) AS mean_consumption,

        MIN(
            {TARGET_COLUMN}
        ) AS min_consumption,

        MAX(
            {TARGET_COLUMN}
        ) AS max_consumption

    FROM energy_features

    GROUP BY year

    ORDER BY year
    """
)


yearly_sql.show(
    truncate=False
)

+----+-----+------------------+---------------+---------------+
|year|n    |mean_consumption  |min_consumption|max_consumption|
+----+-----+------------------+---------------+---------------+
|2013|16514|55300.30050260385 |29632.0        |88434.0        |
|2014|17520|52780.098002283106|29477.0        |82607.0        |
|2015|17520|54010.58829908676 |29590.0        |91934.0        |
|2016|17568|54682.67344034609 |30545.0        |88731.0        |
|2017|17520|54683.09688926941 |30184.0        |94236.0        |
|2018|17520|54282.72168949772 |30475.0        |96272.0        |
|2019|17520|53707.09046803653 |30759.0        |88453.0        |
|2020|17568|50840.09218465391 |29124.0        |83371.0        |
|2021|17520|53490.016324200915|30130.0        |88416.0        |
|2022|17520|51385.32688356165 |30845.0        |86978.0        |
|2023|17520|49723.43396118721 |29798.0        |83757.0        |
|2024|17568|50101.14981785064 |30348.0        |85370.0        |
|2025|17517|50668.43934463664 |30306.0  

In [220]:
# ==========================================
# MONTHLY AGGREGATION
# ==========================================

monthly_sql = spark.sql(
    f"""
    SELECT
        month,

        AVG(
            {TARGET_COLUMN}
        ) AS mean_consumption,

        STDDEV(
            {TARGET_COLUMN}
        ) AS std_consumption,

        COUNT(*) AS n

    FROM energy_features

    GROUP BY month

    ORDER BY month
    """
)


monthly_sql.show(
    12,
    truncate=False,
)

+-----+------------------+-----------------+-----+
|month|mean_consumption  |std_consumption  |n    |
+-----+------------------+-----------------+-----+
|1    |68401.96766277675 |8843.79858573722 |18338|
|2    |65641.80091961853 |8535.236778828705|17616|
|3    |58619.13262510339 |7921.024195850597|19344|
|4    |50207.94241452991 |7177.499847368386|18720|
|5    |45051.30536600496 |6044.718100020126|19344|
|6    |44460.80977564103 |6065.083774086564|18720|
|7    |44952.32190860215 |6152.684479436077|19344|
|8    |42068.3163254756  |5518.768596036193|19344|
|9    |44817.58322649573 |5970.289296848803|18720|
|10   |48433.57578577337 |6820.687395845894|19344|
|11   |57877.688408119655|8836.521040175267|18720|
|12   |63770.171397549246|8239.92178492338 |19341|
+-----+------------------+-----------------+-----+



In [221]:
# ==========================================
# WEEKDAY VS WEEKEND
# ==========================================

weekend_sql = spark.sql(
    f"""
    SELECT
        is_weekend,

        AVG(
            {TARGET_COLUMN}
        ) AS mean_consumption,

        STDDEV(
            {TARGET_COLUMN}
        ) AS std_consumption,

        COUNT(*) AS n

    FROM energy_features

    GROUP BY is_weekend

    ORDER BY is_weekend
    """
)


weekend_sql.show(
    truncate=False
)

+----------+------------------+------------------+------+
|is_weekend|mean_consumption  |std_consumption   |n     |
+----------+------------------+------------------+------+
|0         |54478.297381165365|11473.244514225835|162095|
|1         |48359.60360339506 |10604.15033111562 |64800 |
+----------+------------------+------------------+------+



In [222]:
# ==========================================
# TOP CONSUMPTION OBSERVATIONS
# ==========================================

top_consumption_sql = spark.sql(
    f"""
    SELECT
        {TIMESTAMP_COLUMN},
        {TARGET_COLUMN},

        lag_24h,
        lag_7d,

        rolling_mean_24h,
        rolling_mean_7d

    FROM energy_features

    ORDER BY
        {TARGET_COLUMN} DESC

    LIMIT 10
    """
)


top_consumption_sql.show(
    10,
    truncate=False,
)

+-------------------+--------------+-------+-------+-----------------+-----------------+
|timestamp_utc      |consumption_mw|lag_24h|lag_7d |rolling_mean_24h |rolling_mean_7d  |
+-------------------+--------------+-------+-------+-----------------+-----------------+
|2018-02-28 18:30:00|96272.0       |94202.0|81843.0|89519.20833333333|79039.42559523809|
|2018-02-28 18:00:00|95987.0       |93147.0|81221.0|89460.04166666667|78995.47916666667|
|2018-02-28 11:00:00|95752.0       |92720.0|78914.0|88258.16666666667|78269.53869047618|
|2018-02-28 09:00:00|95647.0       |92553.0|79738.0|87989.14583333333|78074.79166666667|
|2018-02-28 10:30:00|95646.0       |92408.0|78882.0|88190.70833333333|78219.64583333333|
|2018-02-28 09:30:00|95555.0       |92394.0|79441.0|88053.60416666667|78122.13988095238|
|2018-02-28 08:30:00|95539.0       |92741.0|79913.0|87930.85416666667|78028.28571428571|
|2018-02-28 10:00:00|95420.0       |92000.0|78772.0|88119.45833333333|78170.09821428571|
|2018-02-28 12:00:00|

In [223]:
# ==========================================
# SQL WINDOW FUNCTION
# ==========================================

sql_window_demo = spark.sql(
    f"""
    SELECT

        {TIMESTAMP_COLUMN},

        {TARGET_COLUMN},

        LAG(
            {TARGET_COLUMN},
            48
        )
        OVER (
            ORDER BY
                {TIMESTAMP_COLUMN}
        ) AS sql_lag_24h

    FROM energy_features

    ORDER BY
        {TIMESTAMP_COLUMN}

    LIMIT 60
    """
)


sql_window_demo.show(
    60,
    truncate=False,
)

+-------------------+--------------+-----------+
|timestamp_utc      |consumption_mw|sql_lag_24h|
+-------------------+--------------+-----------+
|2013-01-21 23:00:00|76236.0       |NULL       |
|2013-01-21 23:30:00|74650.0       |NULL       |
|2013-01-22 00:00:00|71986.0       |NULL       |
|2013-01-22 00:30:00|73192.0       |NULL       |
|2013-01-22 01:00:00|72637.0       |NULL       |
|2013-01-22 01:30:00|72436.0       |NULL       |
|2013-01-22 02:00:00|70376.0       |NULL       |
|2013-01-22 02:30:00|69090.0       |NULL       |
|2013-01-22 03:00:00|68165.0       |NULL       |
|2013-01-22 03:30:00|67932.0       |NULL       |
|2013-01-22 04:00:00|68240.0       |NULL       |
|2013-01-22 04:30:00|70228.0       |NULL       |
|2013-01-22 05:00:00|71996.0       |NULL       |
|2013-01-22 05:30:00|76064.0       |NULL       |
|2013-01-22 06:00:00|79496.0       |NULL       |
|2013-01-22 06:30:00|82743.0       |NULL       |
|2013-01-22 07:00:00|84210.0       |NULL       |
|2013-01-22 07:30:00

In [224]:
# ==========================================
# SQL LAG VALIDATION
# ==========================================

sql_lag_validation = spark.sql(
    f"""
    SELECT

        {TIMESTAMP_COLUMN},

        lag_24h,

        LAG(
            {TARGET_COLUMN},
            48
        )
        OVER (
            ORDER BY
                {TIMESTAMP_COLUMN}
        ) AS sql_lag_24h

    FROM energy_features
    """
)

In [225]:
lag_difference_df = (
    sql_lag_validation

    .filter(
        F.col(
            "lag_24h"
        ).isNotNull()
        &
        F.col(
            "sql_lag_24h"
        ).isNotNull()
    )

    .withColumn(
        "difference",

        F.abs(
            F.col(
                "lag_24h"
            )
            - F.col(
                "sql_lag_24h"
            )
        )
    )
)

In [226]:
max_lag_difference = (
    lag_difference_df

    .agg(
        F.max(
            "difference"
        ).alias(
            "max_difference"
        )
    )

    .collect()[0][
        "max_difference"
    ]
)


print(
    "Différence maximale des lags :",
    max_lag_difference
)

Différence maximale des lags : 0.0


In [227]:
# ==========================================
# SPARK SQL BENCHMARK
# ==========================================

sql_aggregation_start = (
    time.perf_counter()
)


sql_result = (
    spark.sql(
        f"""
        SELECT

            year,
            month,

            AVG(
                {TARGET_COLUMN}
            ) AS mean_consumption,

            STDDEV(
                {TARGET_COLUMN}
            ) AS std_consumption,

            COUNT(*) AS n

        FROM energy_features

        GROUP BY
            year,
            month
        """
    )

    .collect()
)


sql_aggregation_seconds = (
    time.perf_counter()
    - sql_aggregation_start
)


print(
    "Première agrégation Spark SQL :",
    f"{sql_aggregation_seconds:.3f} s"
)

Première agrégation Spark SQL : 0.245 s


In [228]:
# ==========================================
# SECOND SQL BENCHMARK
# ==========================================

second_sql_start = (
    time.perf_counter()
)


second_sql_result = (
    spark.sql(
        f"""
        SELECT

            year,
            month,

            AVG(
                {TARGET_COLUMN}
            ) AS mean_consumption,

            STDDEV(
                {TARGET_COLUMN}
            ) AS std_consumption,

            COUNT(*) AS n

        FROM energy_features

        GROUP BY
            year,
            month
        """
    )

    .collect()
)


second_sql_seconds = (
    time.perf_counter()
    - second_sql_start
)


print(
    "Deuxième agrégation Spark SQL :",
    f"{second_sql_seconds:.3f} s"
)

Deuxième agrégation Spark SQL : 0.215 s


In [229]:
# ==========================================
# SQL REPEAT SPEEDUP
# ==========================================

if second_sql_seconds > 0:

    sql_speedup = (
        sql_aggregation_seconds
        / second_sql_seconds
    )

else:

    sql_speedup = np.nan


print(
    "Speed-up deuxième requête :",
    f"{sql_speedup:.2f}x"
)

Speed-up deuxième requête : 1.14x


In [230]:
# ==========================================
# SQL EXECUTION PLAN
# ==========================================

sql_plan_df = spark.sql(
    f"""
    SELECT

        year,
        month,

        AVG(
            {TARGET_COLUMN}
        ) AS mean_consumption,

        COUNT(*) AS n

    FROM energy_features

    GROUP BY
        year,
        month
    """
)


sql_plan_df.explain(
    mode="formatted"
)

== Physical Plan ==
AdaptiveSparkPlan (30)
+- HashAggregate (29)
   +- HashAggregate (28)
      +- InMemoryTableScan (1)
            +- InMemoryRelation (2)
                  +- AdaptiveSparkPlan (27)
                     +- == Final Plan ==
                        ResultQueryStage (19)
                        +- * Project (18)
                           +- * Filter (17)
                              +- Window (16)
                                 +- * Sort (15)
                                    +- ShuffleQueryStage (14), Statistics(sizeInBytes=50.3 MiB, rowCount=2.27E+5)
                                       +- Exchange (13)
                                          +- * Project (12)
                                             +- * Project (11)
                                                +- TableCacheQueryStage (10), Statistics(sizeInBytes=39.9 MiB, rowCount=2.27E+5)
                                                   +- InMemoryTableScan (3)
                                   

In [231]:
# ==========================================
# AGGREGATION PARTITIONS
# ==========================================

print(
    "Partitions dataset source :",
    energy_features_cached
    .rdd
    .getNumPartitions()
)


print(
    "Partitions résultat agrégation :",
    sql_plan_df
    .rdd
    .getNumPartitions()
)

Partitions dataset source : 1
Partitions résultat agrégation : 1


In [232]:
# ==========================================
# STEP 4 PERFORMANCE SUMMARY
# ==========================================

step4_performance = pd.DataFrame({

    "Operation": [
        "First SQL aggregation",
        "Second SQL aggregation",
    ],

    "Seconds": [
        sql_aggregation_seconds,
        second_sql_seconds,
    ],
})


display(
    step4_performance
)

,Operation,Seconds
0,First SQL aggregation,0.245066
1,Second SQL aggregation,0.214638


In [233]:
step4_summary = pd.DataFrame({

    "Indicator": [
        "API / SQL max difference",
        "Lag max difference",
        "First SQL aggregation (s)",
        "Second SQL aggregation (s)",
        "Repeat speed-up",
        "Max year",
    ],

    "Value": [
        max_hourly_difference,
        max_lag_difference,
        sql_aggregation_seconds,
        second_sql_seconds,
        sql_speedup,
        2025,
    ],
})


display(
    step4_summary
)

,Indicator,Value
0,API / SQL max difference,0.000000
1,Lag max difference,0.000000
2,First SQL aggregation (s),0.245066
3,Second SQL aggregation (s),0.214638
4,Repeat speed-up,1.141766
5,Max year,2025.000000


In [234]:
# ==========================================
# STEP 4 VALIDATION
# ==========================================

assert (
    max_hourly_difference
    < 1e-9
)


assert (
    max_lag_difference
    < 1e-9
)


assert (
    sql_check_row[
        "max_year"
    ]
    <= 2025
)


assert (
    len(
        sql_result
    )
    > 0
)


print(
    "Étape 4 : OK"
)

Étape 4 : OK


### Partie C — Analyse et améliorations guidées par la théorie

Spark SQL et l'API DataFrame constituent deux interfaces différentes
vers le même moteur d'exécution.

Une requête SQL et une transformation DataFrame équivalentes sont
converties en plans logiques puis optimisées par Spark.

L'expérience confirme donc que :

$$
Result_{SQL}
\approx
Result_{DataFrame}.
$$

Pour une agrégation distribuée, Spark peut réaliser une première
agrégation dans chaque partition :

$$
A_j =
Aggregate(P_j).
$$

Un shuffle redistribue ensuite les résultats selon les clés du
`GROUP BY`.

L'agrégation finale combine les résultats intermédiaires :

$$
A =
Aggregate(A_1,\ldots,A_P).
$$

Le plan physique fait ainsi apparaître typiquement :

`InMemoryTableScan -> HashAggregate -> Exchange -> HashAggregate`.

Cette architecture est différente des Window Functions globales
étudiées précédemment.

Les fenêtres temporelles globales ont réduit le dataset de 8 partitions
à une seule partition afin de préserver l'ordre temporel.

Les agrégations par groupe peuvent au contraire exploiter plusieurs
partitions avant le shuffle.

Spark est donc davantage adapté aux traitements naturellement
partitionnables qu'aux fenêtres globales appliquées à une unique série
temporelle.

Conclusion :

L'Étape 4 montre que les traitements énergétiques peuvent être exprimés
de manière équivalente avec Spark SQL et l'API DataFrame.

Les agrégations horaires, mensuelles, annuelles et calendaires sont
réalisées directement dans le moteur distribué.

L'analyse du plan physique permet d'identifier les agrégations locales,
les shuffles et les agrégations finales.

Les résultats confirment également la différence entre les traitements
facilement parallélisables et les fenêtres temporelles globales qui
constituent un goulot d'étranglement pour une unique série nationale.

Aucune donnée de 2026 n'est utilisée.

### Interprétation du benchmark SQL

Les deux exécutions de l'agrégation Spark SQL présentent des temps très
proches :

- première exécution : environ 0.445 s ;
- deuxième exécution : environ 0.455 s.

Le speed-up obtenu est :

$$
S \approx 0.978.
$$

Il n'y a donc pas d'accélération mesurable lors de la deuxième
exécution.

Cette observation est cohérente avec le fait que le DataFrame source
était déjà matérialisé dans le cache Spark. Les deux requêtes utilisent
donc directement les données en mémoire.

La différence observée, de l'ordre de quelques millisecondes, est trop
faible pour être interprétée comme une différence significative de
performance.

Les résultats Spark SQL et API DataFrame sont par ailleurs exactement
identiques, de même que les lags temporels reconstruits en SQL.

## Étape 5 — Benchmark contrôlé Pandas, PostgreSQL et Spark

### Partie A — Mise en pratique

Cette étape compare les performances des trois moteurs utilisés dans le
projet :

- Pandas ;
- PostgreSQL ;
- Apache Spark.

Une comparaison correcte nécessite de distinguer :

1. le temps de chargement des données ;
2. le temps de calcul ;
3. le coût de communication ;
4. le coût de parallélisation.

Le premier benchmark compare Pandas et Spark sur des données déjà
chargées en mémoire afin d'isoler autant que possible le coût du calcul.

PostgreSQL sera ensuite évalué directement côté base de données.

L'objectif n'est pas de montrer que Spark est systématiquement plus
rapide, mais d'identifier les situations dans lesquelles chaque
architecture devient pertinente.

In [235]:
# ==========================================
# BENCHMARK CONFIGURATION
# ==========================================

BENCHMARK_RUNS = 5


print(
    "Nombre de répétitions :",
    BENCHMARK_RUNS
)

Nombre de répétitions : 5


In [236]:
# ==========================================
# BUILD COMPARABLE PANDAS DATASET
# ==========================================

min_clean_timestamp = (
    energy_features_cached

    .agg(
        F.min(
            TIMESTAMP_COLUMN
        ).alias(
            "min_timestamp"
        )
    )

    .collect()[0][
        "min_timestamp"
    ]
)


print(
    "Premier timestamp comparable :",
    min_clean_timestamp
)

Premier timestamp comparable : 2013-01-22 00:00:00


In [237]:
energy_pd_benchmark = (
    energy_pd[
        energy_pd[
            TIMESTAMP_COLUMN
        ]
        >= pd.Timestamp(
            min_clean_timestamp,
            tz="UTC"
        )
    ]
    .copy()
)

In [238]:
# ==========================================
# PANDAS CALENDAR FEATURES
# ==========================================

energy_pd_benchmark[
    "year"
] = (
    energy_pd_benchmark[
        TIMESTAMP_COLUMN
    ]
    .dt
    .year
)


energy_pd_benchmark[
    "month"
] = (
    energy_pd_benchmark[
        TIMESTAMP_COLUMN
    ]
    .dt
    .month
)

In [239]:
print(
    "Pandas benchmark rows :",
    len(
        energy_pd_benchmark
    )
)

print(
    "Spark benchmark rows :",
    energy_features_cached.count()
)

Pandas benchmark rows : 226893
Spark benchmark rows : 226895


In [240]:
# ==========================================
# PANDAS BENCHMARK FUNCTION
# ==========================================

def pandas_aggregation_benchmark(
    dataframe
):

    return (
        dataframe

        .groupby(
            [
                "year",
                "month",
            ],
            as_index=False,
        )

        .agg(
            mean_consumption=(
                TARGET_COLUMN,
                "mean",
            ),

            std_consumption=(
                TARGET_COLUMN,
                "std",
            ),

            n=(
                TARGET_COLUMN,
                "size",
            ),
        )
    )

In [241]:
pandas_result = (
    pandas_aggregation_benchmark(
        energy_pd_benchmark
    )
)


display(
    pandas_result.head()
)

,year,month,mean_consumption,std_consumption,n
0,2013,1,73629.718750,7229.082537,480
1,2013,2,73432.066220,6758.296070,1344
2,2013,3,66187.393481,7503.313725,1488
3,2013,4,55720.717361,8252.372893,1440
4,2013,5,49279.794355,6375.295427,1488


In [242]:
# ==========================================
# PANDAS TIMING
# ==========================================

pandas_times = []


for _ in range(
    BENCHMARK_RUNS
):

    start = (
        time.perf_counter()
    )


    _ = (
        pandas_aggregation_benchmark(
            energy_pd_benchmark
        )
    )


    pandas_times.append(
        time.perf_counter()
        - start
    )

In [243]:
print(
    "Pandas times :",
    pandas_times
)

Pandas times : [0.02159350000147242, 0.022205599991139024, 0.019454199995379895, 0.01578929999959655, 0.017096800002036616]


In [244]:
# ==========================================
# SPARK BENCHMARK FUNCTION
# ==========================================

def spark_aggregation_benchmark():

    return (
        energy_features_cached

        .groupBy(
            "year",
            "month",
        )

        .agg(

            F.avg(
                TARGET_COLUMN
            ).alias(
                "mean_consumption"
            ),

            F.stddev(
                TARGET_COLUMN
            ).alias(
                "std_consumption"
            ),

            F.count(
                "*"
            ).alias(
                "n"
            ),
        )

        .collect()
    )

In [245]:
# ==========================================
# SPARK TIMING
# ==========================================

spark_times = []


for _ in range(
    BENCHMARK_RUNS
):

    start = (
        time.perf_counter()
    )


    _ = (
        spark_aggregation_benchmark()
    )


    spark_times.append(
        time.perf_counter()
        - start
    )

In [246]:
print(
    "Spark times :",
    spark_times
)

Spark times : [0.22292410000227392, 0.17183219999424182, 0.1629696000018157, 0.1899349000013899, 0.1862012999918079]


In [247]:
# ==========================================
# BENCHMARK STATISTICS
# ==========================================

pandas_mean = float(
    np.mean(
        pandas_times
    )
)

pandas_median = float(
    np.median(
        pandas_times
    )
)


spark_mean = float(
    np.mean(
        spark_times
    )
)

spark_median = float(
    np.median(
        spark_times
    )
)

In [248]:
benchmark_engine = pd.DataFrame({

    "Engine": [
        "Pandas",
        "Spark",
    ],

    "Mean_seconds": [
        pandas_mean,
        spark_mean,
    ],

    "Median_seconds": [
        pandas_median,
        spark_median,
    ],

    "Min_seconds": [
        min(
            pandas_times
        ),
        min(
            spark_times
        ),
    ],

    "Max_seconds": [
        max(
            pandas_times
        ),
        max(
            spark_times
        ),
    ],
})


display(
    benchmark_engine
)

,Engine,Mean_seconds,Median_seconds,Min_seconds,Max_seconds
0,Pandas,0.019228,0.019454,0.015789,0.022206
1,Spark,0.186772,0.186201,0.162970,0.222924


In [249]:
spark_vs_pandas_ratio = (
    spark_median
    / pandas_median
)


print(
    "Spark / Pandas :",
    f"{spark_vs_pandas_ratio:.2f}x"
)

Spark / Pandas : 9.57x


In [250]:
spark_result_pd = (
    energy_features_cached

    .groupBy(
        "year",
        "month",
    )

    .agg(

        F.avg(
            TARGET_COLUMN
        ).alias(
            "mean_consumption"
        ),

        F.stddev(
            TARGET_COLUMN
        ).alias(
            "std_consumption"
        ),

        F.count(
            "*"
        ).alias(
            "n"
        ),
    )

    .orderBy(
        "year",
        "month",
    )

    .toPandas()
)

In [251]:
pandas_result_sorted = (
    pandas_result

    .sort_values(
        [
            "year",
            "month",
        ]
    )

    .reset_index(
        drop=True
    )
)

In [252]:
spark_result_sorted = (
    spark_result_pd

    .sort_values(
        [
            "year",
            "month",
        ]
    )

    .reset_index(
        drop=True
    )
)

In [253]:
mean_difference = (
    np.abs(
        pandas_result_sorted[
            "mean_consumption"
        ]
        -
        spark_result_sorted[
            "mean_consumption"
        ]
    )
    .max()
)


std_difference = (
    np.abs(
        pandas_result_sorted[
            "std_consumption"
        ]
        -
        spark_result_sorted[
            "std_consumption"
        ]
    )
    .max()
)


count_difference = (
    np.abs(
        pandas_result_sorted[
            "n"
        ]
        -
        spark_result_sorted[
            "n"
        ]
    )
    .max()
)


print(
    "Mean max difference :",
    mean_difference
)

print(
    "STD max difference :",
    std_difference
)

print(
    "Count max difference :",
    count_difference
)

Mean max difference : 7.523988589207875
STD max difference : 13.920165294735853
Count max difference : 2


In [254]:
spark_ingestion_seconds = (
    postgres_load_seconds
    + spark_conversion_seconds
)


print(
    "Coût initial PostgreSQL -> Pandas -> Spark :",
    f"{spark_ingestion_seconds:.3f} s"
)

Coût initial PostgreSQL -> Pandas -> Spark : 6.837 s


In [255]:
# ==========================================
# BREAK-EVEN ANALYSIS
# ==========================================

if spark_median < pandas_median:

    query_gain = (
        pandas_median
        - spark_median
    )

    break_even_queries = (
        spark_conversion_seconds
        / query_gain
    )


    print(
        "Nombre approximatif de requêtes "
        "pour amortir la conversion Spark :",
        f"{break_even_queries:.1f}"
    )

else:

    break_even_queries = np.inf

    print(
        "Pas de break-even sur ce benchmark : "
        "Spark n'est pas plus rapide par requête."
    )

Pas de break-even sur ce benchmark : Spark n'est pas plus rapide par requête.


In [256]:
# ==========================================
# POSTGRESQL MODULE INSPECTION
# ==========================================

import src.database.postgres as postgres_module


postgres_public_objects = [

    name

    for name in dir(
        postgres_module
    )

    if not name.startswith(
        "_"
    )
]


print(
    postgres_public_objects
)

['get_connection', 'load_ml_dataset', 'os', 'pd', 'psycopg']


In [257]:
import inspect


for name in postgres_public_objects:

    obj = getattr(
        postgres_module,
        name
    )

    if inspect.isfunction(
        obj
    ):

        try:

            print(
                name,
                inspect.signature(
                    obj
                )
            )

        except Exception:

            pass

get_connection ()
load_ml_dataset (start, end)


### Partie C — Analyse et améliorations guidées par la théorie

Une comparaison de performances entre Pandas et Spark doit être
contrôlée.

Comparer directement :

$$
T_{Pandas}
$$

à :

$$
T_{Spark}
$$

sans prendre en compte le chargement, le cache et la matérialisation
peut conduire à des conclusions incorrectes.

Pour Spark, le coût total peut être décomposé en :

$$
T_{total}
=
T_{ingestion}
+
T_{shuffle}
+
T_{calcul}
+
T_{communication}.
$$

Pandas possède moins de coûts d'infrastructure :

$$
T_{total}
\approx
T_{chargement}
+
T_{calcul}.
$$

Sur un dataset tenant facilement en mémoire, cette simplicité peut
donner un avantage important à Pandas.

Spark devient particulièrement intéressant lorsque :

- les données deviennent trop volumineuses pour la mémoire d'une seule
  machine ;
- plusieurs partitions peuvent être traitées réellement en parallèle ;
- de nombreuses transformations réutilisent les mêmes données ;
- le traitement est distribué sur plusieurs machines.

Le benchmark doit donc mesurer non seulement le temps d'une opération,
mais également les coûts nécessaires pour atteindre cette opération.

### Premier résultat du benchmark

Sur environ 227 000 observations, l'agrégation Pandas est nettement
plus rapide que l'agrégation Spark.

Les temps médians observés avant correction finale de l'alignement sont
approximativement :

- Pandas : 0.029 s ;
- Spark : 0.295 s.

Le rapport de temps est donc d'environ :

$$
\frac{T_{Spark}}{T_{Pandas}}
\approx
10.2.
$$

Pour cette opération et ce volume de données, Spark est donc environ
dix fois plus lent que Pandas.

Ce résultat s'explique par le fait que les données tiennent facilement
en mémoire sur une seule machine.

Spark doit supporter plusieurs coûts supplémentaires :

- planification du calcul ;
- communication Python/JVM ;
- gestion des partitions ;
- sérialisation ;
- ordonnancement des tâches.

Ces coûts ne sont pas encore compensés par les bénéfices du
parallélisme.

Ce résultat ne signifie pas que Spark est intrinsèquement moins
performant. Il indique que son utilisation n'est pas justifiée par la
vitesse pour le volume actuel du projet.

In [258]:
# ==========================================
# EXACT PANDAS / SPARK ROW ALIGNMENT
# ==========================================

MAX_LOOKBACK = max(
    max(
        LAG_STEPS.values()
    ),
    336,
)


energy_pd_benchmark = (
    energy_pd

    .sort_values(
        TIMESTAMP_COLUMN
    )

    .iloc[
        MAX_LOOKBACK:
    ]

    .copy()
)


energy_pd_benchmark[
    "year"
] = (
    energy_pd_benchmark[
        TIMESTAMP_COLUMN
    ]
    .dt
    .year
)


energy_pd_benchmark[
    "month"
] = (
    energy_pd_benchmark[
        TIMESTAMP_COLUMN
    ]
    .dt
    .month
)


print(
    "Lookback maximal :",
    MAX_LOOKBACK
)

print(
    "Pandas benchmark rows :",
    len(
        energy_pd_benchmark
    )
)

print(
    "Spark benchmark rows :",
    energy_features_cached.count()
)


assert (
    len(
        energy_pd_benchmark
    )
    ==
    energy_features_cached.count()
)


print(
    "Alignement Pandas / Spark : OK"
)

Lookback maximal : 336
Pandas benchmark rows : 226895
Spark benchmark rows : 226895
Alignement Pandas / Spark : OK


In [259]:
print(
    "Spark SQL timezone :",
    spark.conf.get(
        "spark.sql.session.timeZone"
    )
)

print(
    "Pandas timestamp dtype :",
    energy_pd[
        TIMESTAMP_COLUMN
    ].dtype
)

Spark SQL timezone : UTC
Pandas timestamp dtype : datetime64[ns, UTC]


In [260]:
spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

In [261]:
pandas_result = (
    pandas_aggregation_benchmark(
        energy_pd_benchmark
    )
)

In [262]:
pandas_result = (
    pandas_aggregation_benchmark(
        energy_pd_benchmark
    )
)

In [263]:
# ==========================================
# FIND POSTGRESQL ML TABLE
# ==========================================

from src.database.postgres import (
    get_connection,
)


with get_connection() as conn:

    postgres_candidates = pd.read_sql_query(
        """
        SELECT
            table_schema,
            table_name,

            COUNT(*) FILTER (
                WHERE column_name = 'timestamp_utc'
            ) AS has_timestamp,

            COUNT(*) FILTER (
                WHERE column_name = 'consumption_mw'
            ) AS has_consumption

        FROM information_schema.columns

        WHERE table_schema = 'energy'

        GROUP BY
            table_schema,
            table_name

        HAVING
            COUNT(*) FILTER (
                WHERE column_name = 'timestamp_utc'
            ) > 0

            AND

            COUNT(*) FILTER (
                WHERE column_name = 'consumption_mw'
            ) > 0

        ORDER BY
            table_name
        """,
        conn,
    )


display(
    postgres_candidates
)

C:\Users\thiba\AppData\Local\Temp\ipykernel_20428\393597787.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  postgres_candidates = pd.read_sql_query(


,table_schema,table_name,has_timestamp,has_consumption
0,energy,consumption,1,1
1,energy,ml_dataset,1,1


In [264]:
if len(
    postgres_candidates
) == 1:

    POSTGRES_SCHEMA = (
        postgres_candidates.iloc[
            0
        ][
            "table_schema"
        ]
    )

    POSTGRES_TABLE = (
        postgres_candidates.iloc[
            0
        ][
            "table_name"
        ]
    )


    print(
        "Table sélectionnée :",
        f"{POSTGRES_SCHEMA}.{POSTGRES_TABLE}"
    )

else:

    print(
        "Plusieurs tables possibles :"
    )

    display(
        postgres_candidates
    )

Plusieurs tables possibles :


,table_schema,table_name,has_timestamp,has_consumption
0,energy,consumption,1,1
1,energy,ml_dataset,1,1


In [265]:
benchmark_start = (
    energy_pd_benchmark[
        TIMESTAMP_COLUMN
    ].min()
)


benchmark_end = (
    energy_pd_benchmark[
        TIMESTAMP_COLUMN
    ].max()
)


print(
    "Début benchmark :",
    benchmark_start
)

print(
    "Fin benchmark :",
    benchmark_end
)

Début benchmark : 2013-01-21 23:00:00+00:00
Fin benchmark : 2025-12-31 22:00:00+00:00


In [267]:
# ==========================================
# FIND EXACT POSTGRESQL TABLE
# ==========================================

from src.database.postgres import get_connection


EXPECTED_ROWS = len(
    energy_pd_benchmark
)


with get_connection() as conn:

    postgres_candidates = pd.read_sql_query(
        """
        SELECT
            table_schema,
            table_name

        FROM information_schema.columns

        WHERE table_schema = 'energy'

        GROUP BY
            table_schema,
            table_name

        HAVING
            COUNT(*) FILTER (
                WHERE column_name = 'timestamp_utc'
            ) > 0

            AND

            COUNT(*) FILTER (
                WHERE column_name = 'consumption_mw'
            ) > 0

        ORDER BY
            table_name
        """,
        conn,
    )


display(
    postgres_candidates
)

C:\Users\thiba\AppData\Local\Temp\ipykernel_20428\545170823.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  postgres_candidates = pd.read_sql_query(


,table_schema,table_name
0,energy,consumption
1,energy,ml_dataset


In [268]:
# ==========================================
# COUNT ROWS IN EACH CANDIDATE TABLE
# ==========================================

candidate_results = []


with get_connection() as conn:

    for _, row in postgres_candidates.iterrows():

        schema = row[
            "table_schema"
        ]

        table = row[
            "table_name"
        ]


        query = f"""
        SELECT
            COUNT(*) AS n

        FROM
            "{schema}"."{table}"

        WHERE
            "{TIMESTAMP_COLUMN}" >= %s

            AND

            "{TIMESTAMP_COLUMN}" <= %s
        """


        result = pd.read_sql_query(
            query,
            conn,
            params=[
                benchmark_start,
                benchmark_end,
            ],
        )


        candidate_results.append({

            "schema":
                schema,

            "table":
                table,

            "rows":
                int(
                    result.iloc[
                        0
                    ][
                        "n"
                    ]
                ),
        })


candidate_counts = pd.DataFrame(
    candidate_results
)


display(
    candidate_counts
)

C:\Users\thiba\AppData\Local\Temp\ipykernel_20428\660136645.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql_query(


,schema,table,rows
0,energy,consumption,226895
1,energy,ml_dataset,226895


In [269]:
# ==========================================
# SELECT EXACT MATCH
# ==========================================

exact_matches = (
    candidate_counts[
        candidate_counts[
            "rows"
        ]
        == EXPECTED_ROWS
    ]
)


print(
    "Nombre de lignes attendu :",
    EXPECTED_ROWS
)


print(
    "Tables correspondantes :",
    len(
        exact_matches
    )
)

Nombre de lignes attendu : 226895
Tables correspondantes : 2


In [273]:
# ==========================================
# DISAMBIGUATE POSTGRESQL TABLES BY SCHEMA
# ==========================================

expected_columns = set(
    energy_pd.columns
)


schema_comparison = []


with get_connection() as conn:

    for _, row in exact_matches.iterrows():

        schema = row["schema"]
        table = row["table"]

        table_columns_df = pd.read_sql_query(
            """
            SELECT
                column_name

            FROM information_schema.columns

            WHERE
                table_schema = %s
                AND table_name = %s

            ORDER BY
                ordinal_position
            """,
            conn,
            params=[
                schema,
                table,
            ],
        )


        table_columns = set(
            table_columns_df[
                "column_name"
            ]
        )


        common_columns = (
            expected_columns
            & table_columns
        )


        missing_columns = (
            expected_columns
            - table_columns
        )


        extra_columns = (
            table_columns
            - expected_columns
        )


        schema_comparison.append({

            "schema":
                schema,

            "table":
                table,

            "rows":
                int(
                    row["rows"]
                ),

            "n_columns":
                len(
                    table_columns
                ),

            "common_columns":
                len(
                    common_columns
                ),

            "missing_columns":
                len(
                    missing_columns
                ),

            "extra_columns":
                len(
                    extra_columns
                ),

            "exact_schema":
                (
                    table_columns
                    == expected_columns
                ),
        })


schema_comparison_df = pd.DataFrame(
    schema_comparison
)


display(
    schema_comparison_df
)

C:\Users\thiba\AppData\Local\Temp\ipykernel_20428\2659141578.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  table_columns_df = pd.read_sql_query(


,schema,table,rows,n_columns,common_columns,missing_columns,extra_columns,exact_schema
0,energy,consumption,226895,3,2,21,1,False
1,energy,ml_dataset,226895,23,23,0,0,True


In [275]:
# ==========================================
# SELECT TABLE WITH EXACT SCHEMA
# ==========================================

exact_schema_matches = (
    schema_comparison_df[
        schema_comparison_df[
            "exact_schema"
        ]
    ]
)


print(
    "Tables avec schéma exact :",
    len(
        exact_schema_matches
    )
)

Tables avec schéma exact : 1


In [277]:
if len(
    exact_schema_matches
) == 1:

    POSTGRES_SCHEMA = (
        exact_schema_matches
        .iloc[0][
            "schema"
        ]
    )

    POSTGRES_TABLE = (
        exact_schema_matches
        .iloc[0][
            "table"
        ]
    )


    print(
        "Table PostgreSQL sélectionnée :",
        f"{POSTGRES_SCHEMA}.{POSTGRES_TABLE}"
    )


elif len(
    exact_schema_matches
) == 0:

    raise ValueError(
        "Aucune table ne possède exactement "
        "le même schéma que energy_pd."
    )


else:

    print(
        "Plusieurs tables ont exactement "
        "le même nombre de lignes ET le même schéma."
    )

    display(
        exact_schema_matches
    )

Table PostgreSQL sélectionnée : energy.ml_dataset


In [278]:
# ==========================================
# POSTGRESQL AGGREGATION QUERY
# ==========================================

postgres_query = f"""
SELECT

    EXTRACT(
        YEAR FROM {TIMESTAMP_COLUMN}
    )::integer AS year,

    EXTRACT(
        MONTH FROM {TIMESTAMP_COLUMN}
    )::integer AS month,

    AVG(
        {TARGET_COLUMN}
    ) AS mean_consumption,

    STDDEV_SAMP(
        {TARGET_COLUMN}
    ) AS std_consumption,

    COUNT(*) AS n

FROM
    {POSTGRES_SCHEMA}.{POSTGRES_TABLE}

WHERE
    {TIMESTAMP_COLUMN} >= %s

    AND

    {TIMESTAMP_COLUMN} <= %s

GROUP BY
    year,
    month

ORDER BY
    year,
    month
"""

In [279]:
benchmark_start = (
    energy_pd_benchmark[
        TIMESTAMP_COLUMN
    ].min()
)


benchmark_end = (
    energy_pd_benchmark[
        TIMESTAMP_COLUMN
    ].max()
)


print(
    "Début benchmark :",
    benchmark_start
)

print(
    "Fin benchmark :",
    benchmark_end
)

Début benchmark : 2013-01-21 23:00:00+00:00
Fin benchmark : 2025-12-31 22:00:00+00:00


In [282]:
# ==========================================
# POSTGRESQL AGGREGATION QUERY — FIXED
# ==========================================

postgres_query = f"""
SELECT

    EXTRACT(
        YEAR FROM "{TIMESTAMP_COLUMN}"
    )::integer AS year,

    EXTRACT(
        MONTH FROM "{TIMESTAMP_COLUMN}"
    )::integer AS month,

    AVG(
        "{TARGET_COLUMN}"
    ) AS mean_consumption,

    STDDEV_SAMP(
        "{TARGET_COLUMN}"
    ) AS std_consumption,

    COUNT(*) AS n

FROM
    "{POSTGRES_SCHEMA}"."{POSTGRES_TABLE}"

WHERE
    "{TIMESTAMP_COLUMN}" >= %s

    AND

    "{TIMESTAMP_COLUMN}" <= %s

GROUP BY
    1,
    2

ORDER BY
    1,
    2
"""

In [283]:
with get_connection() as conn:

    postgres_result = pd.read_sql_query(
        postgres_query,
        conn,
        params=[
            benchmark_start,
            benchmark_end,
        ],
    )


display(
    postgres_result.head()
)

C:\Users\thiba\AppData\Local\Temp\ipykernel_20428\1362074827.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  postgres_result = pd.read_sql_query(


,year,month,mean_consumption,std_consumption,n
0,2013,1,73637.242739,7215.162372,482
1,2013,2,73432.066220,6758.296070,1344
2,2013,3,66187.393481,7503.313725,1488
3,2013,4,55720.717361,8252.372893,1440
4,2013,5,49279.794355,6375.295427,1488


In [284]:
print(
    "Groupes PostgreSQL :",
    len(
        postgres_result
    )
)

Groupes PostgreSQL : 156


In [285]:
postgres_count_query = f"""
SELECT
    COUNT(*) AS n

FROM
    {POSTGRES_SCHEMA}.{POSTGRES_TABLE}

WHERE
    {TIMESTAMP_COLUMN} >= %s

    AND

    {TIMESTAMP_COLUMN} <= %s
"""

In [286]:
with get_connection() as conn:

    postgres_count = pd.read_sql_query(
        postgres_count_query,
        conn,
        params=[
            benchmark_start,
            benchmark_end,
        ],
    ).iloc[
        0
    ][
        "n"
    ]


print(
    "PostgreSQL rows :",
    postgres_count
)

print(
    "Pandas rows :",
    len(
        energy_pd_benchmark
    )
)

print(
    "Spark rows :",
    energy_features_cached.count()
)

PostgreSQL rows :

C:\Users\thiba\AppData\Local\Temp\ipykernel_20428\3451738143.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  postgres_count = pd.read_sql_query(


 226895
Pandas rows : 226895
Spark rows : 226895


In [287]:
assert (
    postgres_count
    ==
    len(
        energy_pd_benchmark
    )
    ==
    energy_features_cached.count()
)


print(
    "Trois moteurs alignés : OK"
)

Trois moteurs alignés : OK


In [288]:
# ==========================================
# POSTGRESQL TIMING
# ==========================================

postgres_times = []


with get_connection() as conn:

    for _ in range(
        BENCHMARK_RUNS
    ):

        start = (
            time.perf_counter()
        )


        _ = pd.read_sql_query(
            postgres_query,
            conn,
            params=[
                benchmark_start,
                benchmark_end,
            ],
        )


        postgres_times.append(
            time.perf_counter()
            - start
        )

C:\Users\thiba\AppData\Local\Temp\ipykernel_20428\1754163560.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  _ = pd.read_sql_query(


In [289]:
print(
    "PostgreSQL times :",
    postgres_times
)

PostgreSQL times : [0.4057560000073863, 0.24293609999585897, 0.27267819999542553, 0.2272932000050787, 0.24170160001085605]


In [290]:
postgres_mean = float(
    np.mean(
        postgres_times
    )
)


postgres_median = float(
    np.median(
        postgres_times
    )
)


print(
    "PostgreSQL mean :",
    postgres_mean
)

print(
    "PostgreSQL median :",
    postgres_median
)

PostgreSQL mean : 0.2780730200029211
PostgreSQL median : 0.24293609999585897


In [291]:
benchmark_engines = pd.DataFrame({

    "Engine": [
        "Pandas",
        "PostgreSQL",
        "Spark",
    ],

    "Mean_seconds": [
        pandas_mean,
        postgres_mean,
        spark_mean,
    ],

    "Median_seconds": [
        pandas_median,
        postgres_median,
        spark_median,
    ],

    "Min_seconds": [
        min(
            pandas_times
        ),

        min(
            postgres_times
        ),

        min(
            spark_times
        ),
    ],

    "Max_seconds": [
        max(
            pandas_times
        ),

        max(
            postgres_times
        ),

        max(
            spark_times
        ),
    ],
})


display(
    benchmark_engines
)

,Engine,Mean_seconds,Median_seconds,Min_seconds,Max_seconds
0,Pandas,0.019228,0.019454,0.015789,0.022206
1,PostgreSQL,0.278073,0.242936,0.227293,0.405756
2,Spark,0.186772,0.186201,0.162970,0.222924


In [292]:
postgres_result_sorted = (
    postgres_result

    .sort_values(
        [
            "year",
            "month",
        ]
    )

    .reset_index(
        drop=True
    )
)

In [293]:
postgres_mean_difference = float(
    np.abs(

        pandas_result_sorted[
            "mean_consumption"
        ].astype(
            float
        )

        -

        postgres_result_sorted[
            "mean_consumption"
        ].astype(
            float
        )

    ).max()
)


print(
    "Pandas / PostgreSQL mean max difference :",
    postgres_mean_difference
)

Pandas / PostgreSQL mean max difference : 7.523988589207875


In [294]:
step5_summary = pd.DataFrame({

    "Metric": [

        "Rows",

        "Pandas median (s)",

        "PostgreSQL median (s)",

        "Spark median (s)",

        "Spark / Pandas",

        "Pandas / PostgreSQL max difference",

        "Spark / Pandas max difference",

    ],

    "Value": [

        len(
            energy_pd_benchmark
        ),

        pandas_median,

        postgres_median,

        spark_median,

        (
            spark_median
            / pandas_median
        ),

        postgres_mean_difference,

        mean_difference,
    ],
})


display(
    step5_summary
)

,Metric,Value
0,Rows,226895.000000
1,Pandas median (s),0.019454
2,PostgreSQL median (s),0.242936
3,Spark median (s),0.186201
4,Spark / Pandas,9.571265
5,Pandas / PostgreSQL max difference,7.523989
6,Spark / Pandas max difference,7.523989


In [295]:
# ==========================================
# FINAL RESULT ALIGNMENT CHECK
# ==========================================

# Recalcul Pandas sur les 226 895 lignes alignées
pandas_result_final = (
    energy_pd_benchmark
    .groupby(
        ["year", "month"],
        as_index=False,
    )
    .agg(
        mean_consumption=(
            TARGET_COLUMN,
            "mean",
        ),
        std_consumption=(
            TARGET_COLUMN,
            "std",
        ),
        n=(
            TARGET_COLUMN,
            "size",
        ),
    )
    .sort_values(
        ["year", "month"]
    )
    .reset_index(
        drop=True
    )
)


# Recalcul Spark
spark_result_final = (
    energy_features_cached
    .groupBy(
        "year",
        "month",
    )
    .agg(
        F.avg(
            TARGET_COLUMN
        ).alias(
            "mean_consumption"
        ),
        F.stddev(
            TARGET_COLUMN
        ).alias(
            "std_consumption"
        ),
        F.count(
            "*"
        ).alias(
            "n"
        ),
    )
    .orderBy(
        "year",
        "month",
    )
    .toPandas()
)


# PostgreSQL déjà calculé
postgres_result_final = (
    postgres_result
    .sort_values(
        ["year", "month"]
    )
    .reset_index(
        drop=True
    )
)


print(
    "Pandas groups :",
    len(pandas_result_final)
)

print(
    "Spark groups :",
    len(spark_result_final)
)

print(
    "PostgreSQL groups :",
    len(postgres_result_final)
)

Pandas groups : 156
Spark groups : 156
PostgreSQL groups : 156


In [296]:
# ==========================================
# NUMERICAL DIFFERENCES
# ==========================================

diff_pandas_spark = float(
    np.abs(
        pandas_result_final[
            "mean_consumption"
        ]
        -
        spark_result_final[
            "mean_consumption"
        ]
    ).max()
)


diff_pandas_postgres = float(
    np.abs(
        pandas_result_final[
            "mean_consumption"
        ].astype(float)
        -
        postgres_result_final[
            "mean_consumption"
        ].astype(float)
    ).max()
)


print(
    "Pandas / Spark max difference :",
    diff_pandas_spark
)

print(
    "Pandas / PostgreSQL max difference :",
    diff_pandas_postgres
)

Pandas / Spark max difference : 0.0
Pandas / PostgreSQL max difference : 0.0


### Conclusion de l'Étape 5

Sur 226 895 observations, Pandas est le moteur le plus rapide pour
l'agrégation testée.

Le temps médian observé est d'environ :

- Pandas : 0.019 s ;
- Spark : 0.186 s ;
- PostgreSQL : 0.243 s.

Spark est environ 9.6 fois plus lent que Pandas sur cette opération.

Ce résultat confirme que le volume actuel du projet ne nécessite pas
Spark pour les traitements locaux classiques.

PostgreSQL reste néanmoins pertinent pour le stockage, la persistance
et l'accès aux données par l'API.

Spark est conservé comme environnement expérimental pour étudier la
montée en charge et les traitements distribués.

### Partie A — Mise en pratique

## Étape 6 — Simulation de montée en charge

Le dataset réel contient environ 227 000 observations.

Afin d'étudier simplement l'effet d'une augmentation du volume de
données, une version synthétique d'environ un million de lignes est
créée à partir des observations existantes.

L'objectif n'est pas de créer de nouvelles données réalistes, mais
uniquement de tester la montée en charge des moteurs de calcul.

Seuls Pandas et Spark sont comparés dans cette expérience.

In [297]:
# ==========================================
# SCALE-UP DATASET
# ==========================================

TARGET_ROWS = 1_000_000


base_benchmark = (
    energy_pd_benchmark[
        [
            "year",
            "month",
            TARGET_COLUMN,
        ]
    ]
    .copy()
)


repeat_factor = int(
    np.ceil(
        TARGET_ROWS
        / len(base_benchmark)
    )
)


large_pd = (
    pd.concat(
        [
            base_benchmark
        ]
        * repeat_factor,
        ignore_index=True,
    )
    .iloc[
        :TARGET_ROWS
    ]
    .copy()
)


print(
    "Nombre de lignes :",
    len(large_pd)
)

print(
    "Facteur approximatif :",
    len(large_pd)
    / len(base_benchmark)
)

Nombre de lignes : 1000000
Facteur approximatif : 4.407324974106966


In [298]:
# ==========================================
# LARGE PANDAS -> SPARK
# ==========================================

large_spark_start = (
    time.perf_counter()
)


large_spark = (
    spark.createDataFrame(
        large_pd
    )
    .repartition(
        8
    )
    .cache()
)


large_spark_count = (
    large_spark.count()
)


large_spark_conversion_seconds = (
    time.perf_counter()
    - large_spark_start
)


print(
    "Spark rows :",
    large_spark_count
)

print(
    "Préparation Spark :",
    f"{large_spark_conversion_seconds:.3f} s"
)

Spark rows : 1000000
Préparation Spark : 5.359 s


In [299]:
assert (
    len(large_pd)
    ==
    large_spark_count
    ==
    TARGET_ROWS
)


print(
    "Dataset 1M : OK"
)

Dataset 1M : OK


In [300]:
# ==========================================
# PANDAS — 1M ROWS
# ==========================================

SCALE_RUNS = 3

large_pandas_times = []


for _ in range(
    SCALE_RUNS
):

    start = (
        time.perf_counter()
    )


    _ = (
        large_pd

        .groupby(
            [
                "year",
                "month",
            ],
            as_index=False,
        )

        .agg(
            mean_consumption=(
                TARGET_COLUMN,
                "mean",
            ),

            n=(
                TARGET_COLUMN,
                "size",
            ),
        )
    )


    large_pandas_times.append(
        time.perf_counter()
        - start
    )


print(
    large_pandas_times
)

[0.17636040000070352, 0.14434959999925923, 0.14511409999977332]


In [301]:
# ==========================================
# SPARK — 1M ROWS
# ==========================================

large_spark_times = []


for _ in range(
    SCALE_RUNS
):

    start = (
        time.perf_counter()
    )


    _ = (
        large_spark

        .groupBy(
            "year",
            "month",
        )

        .agg(

            F.avg(
                TARGET_COLUMN
            ).alias(
                "mean_consumption"
            ),

            F.count(
                "*"
            ).alias(
                "n"
            ),
        )

        .collect()
    )


    large_spark_times.append(
        time.perf_counter()
        - start
    )


print(
    large_spark_times
)

[0.9241460000048392, 0.5953444000042509, 0.47675750000053085]


In [302]:
# ==========================================
# SCALE-UP SUMMARY
# ==========================================

large_pandas_median = float(
    np.median(
        large_pandas_times
    )
)


large_spark_median = float(
    np.median(
        large_spark_times
    )
)


large_ratio = (
    large_spark_median
    / large_pandas_median
)


step6_summary = pd.DataFrame({

    "Engine": [
        "Pandas",
        "Spark",
    ],

    "Rows": [
        TARGET_ROWS,
        TARGET_ROWS,
    ],

    "Median_seconds": [
        large_pandas_median,
        large_spark_median,
    ],
})


display(
    step6_summary
)


print(
    "Spark / Pandas :",
    f"{large_ratio:.2f}x"
)

,Engine,Rows,Median_seconds
0,Pandas,1000000,0.145114
1,Spark,1000000,0.595344


Spark / Pandas : 4.10x


In [303]:
# ==========================================
# SCALING COMPARISON
# ==========================================

scaling_comparison = pd.DataFrame({

    "Dataset": [
        "Real ~227k",
        "Synthetic 1M",
    ],

    "Rows": [
        len(
            energy_pd_benchmark
        ),
        TARGET_ROWS,
    ],

    "Pandas_median_s": [
        pandas_median,
        large_pandas_median,
    ],

    "Spark_median_s": [
        spark_median,
        large_spark_median,
    ],

    "Spark_over_Pandas": [
        spark_median
        / pandas_median,

        large_ratio,
    ],
})


display(
    scaling_comparison
)

,Dataset,Rows,Pandas_median_s,Spark_median_s,Spark_over_Pandas
0,Real ~227k,226895,0.019454,0.186201,9.571265
1,Synthetic 1M,1000000,0.145114,0.595344,4.102595


### Interprétation de la montée en charge

Lorsque le volume passe d'environ 227 000 à 1 000 000
d'observations, le rapport de temps Spark / Pandas passe de :

$$
9.57
$$

à :

$$
4.10.
$$

Spark reste donc plus lent, mais son désavantage relatif diminue
nettement avec l'augmentation du volume.

Cela suggère que les coûts fixes liés à Spark deviennent progressivement
moins dominants lorsque la taille des données augmente.

Un million de lignes reste cependant un volume suffisamment faible pour
être traité efficacement en mémoire avec Pandas sur la machine utilisée.

Spark n'est donc toujours pas nécessaire pour ce volume, mais la tendance
observée illustre l'intérêt potentiel d'un moteur distribué à plus grande
échelle.

In [304]:
# ==========================================
# CLEANUP
# ==========================================

large_spark.unpersist()


print(
    "Dataset Spark 1M libéré du cache."
)

Dataset Spark 1M libéré du cache.


### Partie C — Analyse et améliorations guidées par la théorie

L'augmentation du nombre d'observations permet d'étudier la manière dont
les coûts de calcul évoluent avec la taille du dataset.

Pour chaque moteur, on peut considérer un temps de calcul :

$$
T(N),
$$

où $N$ représente le nombre d'observations.

Le rapport :

$$
R(N)
=
\frac{T_{Spark}(N)}
{T_{Pandas}(N)}
$$

permet d'observer l'évolution relative des performances.

Si $R(N)$ diminue lorsque $N$ augmente, les coûts fixes de Spark
deviennent progressivement moins importants relativement au temps de
calcul.

Cependant, un million de lignes reste un volume modéré pour Spark et
les données tiennent encore facilement en mémoire sur une machine
personnelle.

Cette expérience constitue donc une simulation de montée en charge et
non une démonstration d'un véritable besoin de calcul distribué.

Conclusion :

Le benchmark de montée en charge étend la comparaison à un million
d'observations.

Il permet d'observer si l'écart de performance entre Pandas et Spark se
réduit lorsque le volume augmente.

Pandas reste la référence pour les traitements locaux tant que les
données tiennent facilement en mémoire.

Spark est conservé pour les scénarios où le volume, le nombre de séries
ou la distribution des calculs deviennent suffisamment importants pour
compenser ses coûts supplémentaires.

## Étape 7 — Influence du nombre de partitions

### Partie A — Mise en pratique

Spark répartit les données entre plusieurs partitions.

Un nombre trop faible de partitions peut limiter le parallélisme,
tandis qu'un nombre trop élevé peut augmenter les coûts de gestion et
de communication.

Une comparaison simple est réalisée avec 1, 4 et 8 partitions.

In [305]:
# ==========================================
# PARTITION BENCHMARK
# ==========================================

PARTITION_OPTIONS = [
    1,
    4,
    8,
]


partition_results = []


for n_partitions in PARTITION_OPTIONS:

    df_part = (
        spark
        .createDataFrame(
            large_pd
        )
        .repartition(
            n_partitions
        )
        .cache()
    )


    # Matérialisation du cache
    df_part.count()


    times = []


    for _ in range(3):

        start = (
            time.perf_counter()
        )


        _ = (
            df_part

            .groupBy(
                "year",
                "month",
            )

            .agg(
                F.avg(
                    TARGET_COLUMN
                ).alias(
                    "mean_consumption"
                ),

                F.count(
                    "*"
                ).alias(
                    "n"
                ),
            )

            .collect()
        )


        times.append(
            time.perf_counter()
            - start
        )


    partition_results.append({

        "Partitions":
            n_partitions,

        "Median_seconds":
            float(
                np.median(
                    times
                )
            ),
    })


    df_part.unpersist()

In [306]:
partition_benchmark = pd.DataFrame(
    partition_results
)


display(
    partition_benchmark
)

,Partitions,Median_seconds
0,1,0.556957
1,4,0.597984
2,8,0.682214


In [307]:
best_partition_row = (
    partition_benchmark

    .sort_values(
        "Median_seconds"
    )

    .iloc[0]
)


BEST_PARTITIONS = int(
    best_partition_row[
        "Partitions"
    ]
)


print(
    "Meilleur nombre de partitions :",
    BEST_PARTITIONS
)

print(
    "Temps médian :",
    f"{best_partition_row['Median_seconds']:.3f} s"
)

Meilleur nombre de partitions : 1
Temps médian : 0.557 s


In [308]:
BEST_PARTITIONS = 1

### Interprétation du partitionnement

Sur le benchmark à un million de lignes, une seule partition obtient le
meilleur temps médian :

- 1 partition : 0.557 s ;
- 4 partitions : 0.598 s ;
- 8 partitions : 0.682 s.

Pour cette opération simple exécutée localement, l'augmentation du nombre
de partitions ajoute davantage de coûts de gestion qu'elle n'apporte de
gain de parallélisme.

Ce résultat est spécifique au volume, au matériel et à l'opération
testée. Il ne constitue pas une règle générale pour Spark.

### Partie C — Analyse et améliorations guidées par la théorie

Le nombre de partitions influence le compromis entre parallélisme et
coût de gestion.

Avec trop peu de partitions, plusieurs cœurs peuvent rester inutilisés.

Avec trop de partitions, Spark doit gérer davantage de tâches et de
communications.

Le nombre optimal dépend donc du volume des données, du matériel et du
type d'opération.

Dans ce projet, le benchmark avec 1, 4 et 8 partitions permet de choisir
une configuration raisonnable sans effectuer une optimisation excessive.

Il faut également distinguer les agrégations distribuables des fenêtres
temporelles globales.

Pour ces dernières, l'expérience précédente a montré :

$$
8\ partitions
\rightarrow
1\ partition,
$$

car une unique série temporelle nationale doit conserver un ordre global.

Conclusion :

Le benchmark montre que le nombre de partitions peut modifier les
performances Spark, mais qu'une optimisation complexe n'est pas
nécessaire pour le volume actuel.

La configuration retenue sera simplement celle présentant le meilleur
temps médian parmi 1, 4 et 8 partitions.

Cette expérience complète l'analyse de la montée en charge avant la
synthèse finale du Bloc 12.

In [309]:
# ==========================================
# BLOCK 12 — PERFORMANCE SUMMARY
# ==========================================

block12_performance = pd.DataFrame({

    "Experiment": [
        "227k — Pandas",
        "227k — PostgreSQL",
        "227k — Spark",
        "1M — Pandas",
        "1M — Spark",
        "1M Spark — 1 partition",
        "1M Spark — 4 partitions",
        "1M Spark — 8 partitions",
    ],

    "Median_seconds": [
        0.019454,
        0.242936,
        0.186201,
        0.145114,
        0.595344,
        0.556957,
        0.597984,
        0.682214,
    ],
})


display(
    block12_performance
)

,Experiment,Median_seconds
0,227k — Pandas,0.019454
1,227k — PostgreSQL,0.242936
2,227k — Spark,0.186201
3,1M — Pandas,0.145114
4,1M — Spark,0.595344
5,1M Spark — 1 partition,0.556957
6,1M Spark — 4 partitions,0.597984
7,1M Spark — 8 partitions,0.682214


In [310]:
block12_summary = pd.DataFrame({

    "Indicator": [
        "Real dataset rows",
        "Spark / Pandas — 227k",
        "Spark / Pandas — 1M",
        "Best partitions — 1M",
        "Global temporal window partitions",
        "Production Spark",
        "2026 used",
    ],

    "Value": [
        226895,
        9.571265,
        4.102595,
        BEST_PARTITIONS,
        "8 -> 1",
        False,
        False,
    ],
})


display(
    block12_summary
)

,Indicator,Value
0,Real dataset rows,226895
1,Spark / Pandas — 227k,9.571265
2,Spark / Pandas — 1M,4.102595
3,Best partitions — 1M,1
4,Global temporal window partitions,8 -> 1
5,Production Spark,False
6,2026 used,False


### Partie C — Analyse et améliorations guidées par la théorie

## Architecture retenue

Les expériences du Bloc 12 ne justifient pas le remplacement du pipeline
actuel par Spark.

L'architecture retenue reste :

$$
\text{PostgreSQL}
\rightarrow
\text{API / stockage / persistance}
$$

et :

$$
\text{Pandas}
\rightarrow
\text{analyse locale et traitements en mémoire}.
$$

Spark est conservé comme environnement expérimental pour :

- le calcul distribué ;
- le partitionnement ;
- les shuffles ;
- les Window Functions ;
- la montée en charge.

Sur environ 227 000 observations, Pandas est nettement plus rapide que
Spark pour l'agrégation étudiée.

Lorsque le volume est porté à un million de lignes, Spark reste plus
lent, mais son désavantage relatif diminue :

$$
9.57
\rightarrow
4.10.
$$

Cette évolution montre que les coûts fixes de Spark deviennent moins
dominants lorsque le volume augmente.

Les Window Functions appliquées à l'unique série temporelle nationale
présentent également une limitation importante :

$$
8\ partitions
\rightarrow
1\ partition.
$$

Une architecture comprenant plusieurs séries indépendantes permettrait
davantage de parallélisme grâce à `partitionBy`.

In [311]:
# ==========================================
# SAVE BLOCK 12 FINAL STATE
# ==========================================

BLOCK12_FINAL_PATH = (
    MODEL_DIR
    / "block12_final_state.joblib"
)


block12_final_state = {

    "real_rows":
        226895,

    "pandas_median_227k":
        0.019454,

    "postgresql_median_227k":
        0.242936,

    "spark_median_227k":
        0.186201,

    "spark_pandas_ratio_227k":
        9.571265,

    "pandas_median_1m":
        0.145114,

    "spark_median_1m":
        0.595344,

    "spark_pandas_ratio_1m":
        4.102595,

    "best_partitions":
        BEST_PARTITIONS,

    "global_window_partitions":
        {
            "before": 8,
            "after": 1,
        },

    "spark_production":
        False,

    "postgresql_production":
        True,

    "uses_2026":
        False,
}


joblib.dump(
    block12_final_state,
    BLOCK12_FINAL_PATH,
)


print(
    "Sauvegardé :",
    BLOCK12_FINAL_PATH
)

Sauvegardé : C:\Users\thiba\OneDrive\Bureau\Projet energie\models\block12_final_state.joblib


In [312]:
# ==========================================
# FINAL VALIDATION
# ==========================================

loaded_block12 = joblib.load(
    BLOCK12_FINAL_PATH
)


assert (
    loaded_block12[
        "real_rows"
    ]
    == 226895
)

assert (
    loaded_block12[
        "best_partitions"
    ]
    == 1
)

assert (
    loaded_block12[
        "spark_production"
    ]
    is False
)

assert (
    loaded_block12[
        "uses_2026"
    ]
    is False
)


print(
    "Bloc 12 : validation OK"
)

Bloc 12 : validation OK


## Conclusion du Bloc 12

Le Bloc 12 a permis d'introduire Apache Spark sans complexifier
inutilement l'architecture principale du projet.

Le dataset énergétique réel contient environ 227 000 observations. À
cette échelle, Pandas reste largement plus rapide pour les agrégations
locales étudiées.

Le benchmark donne environ :

- Pandas : 0.019 s ;
- Spark : 0.186 s ;
- PostgreSQL : 0.243 s.

Spark est ainsi environ 9.6 fois plus lent que Pandas sur le dataset
réel.

Lorsque le volume est simulé à un million d'observations, ce rapport
diminue à environ 4.1. Spark reste plus lent, mais son coût relatif
diminue avec l'augmentation du volume.

L'étude des partitions montre également que, sur cette machine et cette
opération, une seule partition est plus rapide que 4 ou 8 partitions.

Enfin, les fenêtres temporelles globales ont montré une limite importante
du calcul distribué sur une unique série :

$$
8\ partitions
\rightarrow
1\ partition.
$$

Le choix final est donc de conserver PostgreSQL et Pandas dans
l'architecture principale et Spark comme laboratoire de montée en charge
et de calcul distribué.

L'année 2026 reste totalement réservée au test final.